In [ ]:
!pip install kaggle -q

# Paste your actual username and key below (replace the placeholder text)
kaggle_token = {"username": "threeno26", "key": "KGAT_efb671c5e70491cb6a0133b4ef1ea51b"}

import json
import os

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_token, f)

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d garymk/movielens-25m-dataset
!unzip -o -q movielens-25m-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/garymk/movielens-25m-dataset
License(s): unknown
100% 258M/258M [00:02<00:00, 90.8MB/s]



In [ ]:
import pandas as pd

# Check what's actually inside the ml-25m folder
!ls -lh ml-25m

# Load from the correct subfolder
ratings = pd.read_csv('ml-25m/ratings.csv')
movies = pd.read_csv('ml-25m/movies.csv')

print("Ratings shape:", ratings.shape)
print("Movies shape:", movies.shape)
print("Unique users:", ratings['userId'].nunique())
print("Unique movies in ratings:", ratings['movieId'].nunique())
print("Rating range:", ratings['rating'].min(), "-", ratings['rating'].max())
print("Timestamp range:", pd.to_datetime(ratings['timestamp'], unit='s').min(),
      "to", pd.to_datetime(ratings['timestamp'], unit='s').max())

ratings.head()

total 1.1G
-rw-r--r-- 1 root root 416M Sep 13  2021 genome-scores.csv
-rw-r--r-- 1 root root  18K Sep 13  2021 genome-tags.csv
-rw-r--r-- 1 root root 1.4M Sep 13  2021 links.csv
-rw-r--r-- 1 root root 2.9M Sep 13  2021 movies.csv
-rw-r--r-- 1 root root 647M Sep 13  2021 ratings.csv
-rw-r--r-- 1 root root  11K Sep 13  2021 README.txt
-rw-r--r-- 1 root root  38M Sep 13  2021 tags.csv
Ratings shape: (25000095, 4)
Movies shape: (62423, 3)
Unique users: 162541
Unique movies in ratings: 59047
Rating range: 0.5 - 5.0
Timestamp range: 1995-01-09 11:46:49 to 2019-11-21 09:15:03


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [ ]:
# Convert Unix timestamp into an actual datetime column — makes it human-readable and usable for time-based operations
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')

# Sort by time — required before we can do a proper temporal split
ratings = ratings.sort_values('datetime').reset_index(drop=True)

# Define split points: use the earliest ~80% of interactions for training,
# next ~10% for validation, final ~10% for testing — this mimics "the future is unseen"
train_cutoff = ratings['datetime'].quantile(0.80)
val_cutoff = ratings['datetime'].quantile(0.90)

train = ratings[ratings['datetime'] <= train_cutoff]
val = ratings[(ratings['datetime'] > train_cutoff) & (ratings['datetime'] <= val_cutoff)]
test = ratings[ratings['datetime'] > val_cutoff]

print("Train:", train.shape, train['datetime'].min(), "to", train['datetime'].max())
print("Val:", val.shape, val['datetime'].min(), "to", val['datetime'].max())
print("Test:", test.shape, test['datetime'].min(), "to", test['datetime'].max())

Train: (20000076, 5) 1995-01-09 11:46:49 to 2016-06-25 06:49:57
Val: (2500009, 5) 2016-06-25 06:50:00 to 2018-01-04 02:44:50
Test: (2500010, 5) 2018-01-04 02:45:08 to 2019-11-21 09:15:03


In [ ]:
# Compute popularity ONLY from the training set — never from val/test,
# since using future data here would be leakage, same principle as the temporal split
movie_popularity = train.groupby('movieId').agg(
    rating_count=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

# Sort by how many times a movie was rated — more ratings = more "popular"
movie_popularity = movie_popularity.sort_values('rating_count', ascending=False).reset_index(drop=True)

# This is our baseline recommendation list: the same top-N movies for EVERY user, regardless of their taste
top_popular_movies = movie_popularity.head(100)['movieId'].tolist()

print("Top 10 most popular movies (by rating count):")
print(movie_popularity.head(10))

Top 10 most popular movies (by rating count):
   movieId  rating_count  avg_rating
0      356         67782    4.035127
1      296         67166    4.176577
2      318         65710    4.429798
3      593         63349    4.157524
4      480         57843    3.661636
5      260         56257    4.156336
6     2571         56149    4.165542
7      110         52275    4.024362
8      527         50724    4.277768
9      589         50400    3.935972


In [ ]:
# Compute popularity ONLY from the training set — never from val/test,
# since using future data here would be leakage, same principle as the temporal split
movie_popularity = train.groupby('movieId').agg(
    rating_count=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

# Sort by how many times a movie was rated — more ratings = more "popular"
movie_popularity = movie_popularity.sort_values('rating_count', ascending=False).reset_index(drop=True)

# This is our baseline recommendation list: the same top-N movies for EVERY user, regardless of their taste
top_popular_movies = movie_popularity.head(100)['movieId'].tolist()

print("Top 10 most popular movies (by rating count):")
print(movie_popularity.head(10))

Top 10 most popular movies (by rating count):
   movieId  rating_count  avg_rating
0      356         67782    4.035127
1      296         67166    4.176577
2      318         65710    4.429798
3      593         63349    4.157524
4      480         57843    3.661636
5      260         56257    4.156336
6     2571         56149    4.165542
7      110         52275    4.024362
8      527         50724    4.277768
9      589         50400    3.935972


In [ ]:
!pip install implicit -q

import scipy.sparse as sparse
from implicit.als import AlternatingLeastSquares

# ALS needs the interaction data as a sparse matrix, not a flat dataframe.
# We need every user and movie mapped to a simple integer index (0, 1, 2, ...)
# because that's what the matrix rows/columns represent internally.
user_ids = train['userId'].unique()
movie_ids = train['movieId'].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}
idx_to_movie = {i: m for m, i in movie_to_idx.items()}

train_user_idx = train['userId'].map(user_to_idx)
train_movie_idx = train['movieId'].map(movie_to_idx)

# Build the sparse user-item matrix: rows = users, columns = movies,
# values = ratings (used here as an implicit "strength of interaction" signal)
user_item_matrix = sparse.csr_matrix(
    (train['rating'], (train_user_idx, train_movie_idx)),
    shape=(len(user_ids), len(movie_ids))
)

print("User-item matrix shape:", user_item_matrix.shape)
print("Non-zero entries (actual interactions):", user_item_matrix.nnz)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 56.7 MB/s eta 0:00:00
User-item matrix shape: (137883, 34461)
Non-zero entries (actual interactions): 20000076


In [ ]:
# ALS expects a slightly different orientation: item-user, not user-item,
# depending on the implicit library version — we build it to match what AlternatingLeastSquares.fit() expects
model = AlternatingLeastSquares(
    factors=50,        # size of each user/movie's "taste profile" — 50 hidden numbers per user/movie
    regularization=0.01,  # prevents the model from overfitting to noise in the training data
    iterations=15       # how many passes the algorithm makes to refine its numbers
)

# Train the model on the user-item matrix we built
model.fit(user_item_matrix)

print("Model trained.")
print("User factors shape:", model.user_factors.shape)  # one row of 50 numbers per user
print("Item factors shape:", model.item_factors.shape)   # one row of 50 numbers per movie

/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

Model trained.
User factors shape: (137883, 50)
Item factors shape: (34461, 50)


In [ ]:
!pip install faiss-cpu -q

import faiss
import numpy as np

# FAISS needs float32 arrays specifically — the item factors from implicit
# might already be float32, but we cast explicitly to be safe
item_vectors = model.item_factors.astype('float32')

# Build a simple FAISS index: this stores all movie vectors in a structure
# optimized for fast "which vectors are closest to this one?" searches
dimension = item_vectors.shape[1]  # should be 50, matching our factors setting
index = faiss.IndexFlatIP(dimension)  # IP = Inner Product, matches how ALS scores work
index.add(item_vectors)

print("FAISS index built.")
print("Number of vectors indexed:", index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.4 MB/s eta 0:00:00
FAISS index built.
Number of vectors indexed: 34461


In [ ]:
def get_candidates(user_id, k=100):
    """
    Given a real userId, return the top-k candidate movieIds using ALS + FAISS.
    Falls back to the popularity baseline if the user isn't in the training set (cold-start).
    """
    # Cold-start check: if this user has no ALS profile, use the popularity fallback
    if user_id not in user_to_idx:
        return top_popular_movies[:k]

    # Get this user's 50-number taste profile
    user_idx = user_to_idx[user_id]
    user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)

    # Ask FAISS: which k movie vectors are closest to this user's vector?
    scores, indices = index.search(user_vector, k)

    # FAISS returns internal matrix positions — convert back to real movieIds
    candidate_movie_ids = [idx_to_movie[i] for i in indices[0]]

    return candidate_movie_ids


# Quick test on a real user from your training data
test_user = train['userId'].iloc[0]
candidates = get_candidates(test_user, k=100)

print(f"Top 10 candidates for user {test_user}:")
print(candidates[:10])

# Also test the cold-start path with a fake/nonexistent user ID
fake_user_candidates = get_candidates(999999999, k=100)
print(f"\nCold-start fallback test (fake user):")
print(fake_user_candidates[:10])

Top 10 candidates for user 2262:
[np.int64(1), np.int64(32), np.int64(608), np.int64(780), np.int64(733), np.int64(648), np.int64(36), np.int64(95), np.int64(736), np.int64(260)]

Cold-start fallback test (fake user):
[356, 296, 318, 593, 480, 260, 2571, 110, 527, 589]


In [ ]:
def recall_at_k(test_df, k=100, sample_users=2000):
    """
    Measures: for users in the test set, what fraction of their
    actual future ratings appear in our top-k candidate list?
    Sampling a subset of users for speed, since testing all ~150K+ users would be slow.
    """
    # Only evaluate users who also appeared in training (ALS knows them)
    # We'll handle cold-start users separately later if needed
    test_users = test_df['userId'].unique()
    known_test_users = [u for u in test_users if u in user_to_idx]

    # Randomly sample a subset for speed
    sampled_users = np.random.choice(known_test_users, size=min(sample_users, len(known_test_users)), replace=False)

    hits = 0
    total = 0

    for user_id in sampled_users:
        # What did this user actually rate in the test period? (ground truth)
        actual_movies = set(test_df[test_df['userId'] == user_id]['movieId'].tolist())

        if len(actual_movies) == 0:
            continue

        # What did our retrieval stage predict for this user?
        candidates = set(get_candidates(user_id, k=k))

        # How many of their actual future ratings did we successfully capture?
        hits += len(actual_movies & candidates)
        total += len(actual_movies)

    return hits / total if total > 0 else 0


recall_score = recall_at_k(test, k=100, sample_users=2000)
print(f"Recall@100: {recall_score:.4f}")

Recall@100: 0.0395


In [ ]:
def recall_at_k_baseline(test_df, k=100, sample_users=2000):
    """
    Same recall calculation, but using the popularity baseline for every user
    instead of ALS+FAISS — this is our comparison point.
    """
    test_users = test_df['userId'].unique()
    sampled_users = np.random.choice(test_users, size=min(sample_users, len(test_users)), replace=False)

    baseline_candidates = set(top_popular_movies[:k])

    hits = 0
    total = 0

    for user_id in sampled_users:
        actual_movies = set(test_df[test_df['userId'] == user_id]['movieId'].tolist())
        if len(actual_movies) == 0:
            continue

        hits += len(actual_movies & baseline_candidates)
        total += len(actual_movies)

    return hits / total if total > 0 else 0


baseline_recall = recall_at_k_baseline(test, k=100, sample_users=2000)
print(f"Popularity Baseline Recall@100: {baseline_recall:.4f}")
print(f"ALS+FAISS Recall@100: {recall_score:.4f}")

Popularity Baseline Recall@100: 0.1212
ALS+FAISS Recall@100: 0.0395


In [ ]:
# Only keep ratings that represent genuine liking — 3.5 and above
# This is the fix: we're removing the "noise" of disliked movies from what ALS learns from
positive_train = train[train['rating'] >= 3.5].copy()

print("Original training interactions:", len(train))
print("Positive-only training interactions:", len(positive_train))
print(f"Kept {len(positive_train) / len(train) * 100:.1f}% of original ratings")

# Rebuild the user/movie ID mappings — using positive_train now, not train,
# since some users/movies might have ONLY had low ratings and would drop out entirely
user_ids = positive_train['userId'].unique()
movie_ids = positive_train['movieId'].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}
idx_to_movie = {i: m for m, i in movie_to_idx.items()}

train_user_idx = positive_train['userId'].map(user_to_idx)
train_movie_idx = positive_train['movieId'].map(movie_to_idx)

# Rebuild the matrix using ONLY positive interactions
user_item_matrix = sparse.csr_matrix(
    (positive_train['rating'], (train_user_idx, train_movie_idx)),
    shape=(len(user_ids), len(movie_ids))
)

print("New user-item matrix shape:", user_item_matrix.shape)
print("Non-zero entries:", user_item_matrix.nnz)

Original training interactions: 20000076
Positive-only training interactions: 12306450
Kept 61.5% of original ratings
New user-item matrix shape: (137759, 28126)
Non-zero entries: 12306450


In [ ]:
# Retrain ALS on the filtered (positive-only) matrix
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=15
)
model.fit(user_item_matrix)

print("Model retrained on positive-only interactions.")
print("User factors shape:", model.user_factors.shape)
print("Item factors shape:", model.item_factors.shape)

# Rebuild the FAISS index using the new item factors
item_vectors = model.item_factors.astype('float32')
dimension = item_vectors.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(item_vectors)

print("FAISS index rebuilt.")
print("Number of vectors indexed:", index.ntotal)

  0%|          | 0/15 [00:00<?, ?it/s]

Model retrained on positive-only interactions.
User factors shape: (137759, 50)
Item factors shape: (28126, 50)
FAISS index rebuilt.
Number of vectors indexed: 28126


In [ ]:
recall_score = recall_at_k(test, k=100, sample_users=2000)
baseline_recall = recall_at_k_baseline(test, k=100, sample_users=2000)

print(f"Popularity Baseline Recall@100: {baseline_recall:.4f}")
print(f"ALS+FAISS Recall@100 (after fix): {recall_score:.4f}")

Popularity Baseline Recall@100: 0.1193
ALS+FAISS Recall@100 (after fix): 0.0375


In [ ]:
# Check: how many of the movies actually liked (3.5+) by test users in the future
# are even POSSIBLE for us to retrieve, given our shrunk movie universe?
test_positive = test[test['rating'] >= 3.5]
test_positive_movies = set(test_positive['movieId'].unique())
retrievable_movies = set(idx_to_movie.values())  # the 28,126 movies ALS can recommend

unreachable = test_positive_movies - retrievable_movies
print(f"Movies genuinely liked in test set: {len(test_positive_movies)}")
print(f"Movies our system could NEVER retrieve: {len(unreachable)}")
print(f"Percent structurally unreachable: {len(unreachable) / len(test_positive_movies) * 100:.1f}%")

Movies genuinely liked in test set: 36844
Movies our system could NEVER retrieve: 15997
Percent structurally unreachable: 43.4%


In [ ]:
# Use ALL users and movies from the original training set — nothing gets excluded structurally
user_ids = train['userId'].unique()
movie_ids = train['movieId'].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}
idx_to_movie = {i: m for m, i in movie_to_idx.items()}

# But only INCLUDE positive ratings as actual matrix entries — same signal-cleaning idea as before,
# just without deleting movies/users that had only negative ratings
positive_train = train[train['rating'] >= 3.5].copy()

train_user_idx = positive_train['userId'].map(user_to_idx)
train_movie_idx = positive_train['movieId'].map(movie_to_idx)

# The matrix SHAPE now covers all original users/movies, but only positive ratings fill in values
user_item_matrix = sparse.csr_matrix(
    (positive_train['rating'], (train_user_idx, train_movie_idx)),
    shape=(len(user_ids), len(movie_ids))
)

print("User-item matrix shape:", user_item_matrix.shape)
print("Non-zero entries (positive interactions only):", user_item_matrix.nnz)

User-item matrix shape: (137883, 34461)
Non-zero entries (positive interactions only): 12306450


In [ ]:
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=15
)
model.fit(user_item_matrix)

print("Model retrained on corrected matrix.")
print("User factors shape:", model.user_factors.shape)
print("Item factors shape:", model.item_factors.shape)

item_vectors = model.item_factors.astype('float32')
dimension = item_vectors.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(item_vectors)

print("FAISS index rebuilt.")
print("Number of vectors indexed:", index.ntotal)

  0%|          | 0/15 [00:00<?, ?it/s]

Model retrained on corrected matrix.
User factors shape: (137883, 50)
Item factors shape: (34461, 50)
FAISS index rebuilt.
Number of vectors indexed: 34461


In [ ]:
recall_score = recall_at_k(test, k=100, sample_users=2000)
baseline_recall = recall_at_k_baseline(test, k=100, sample_users=2000)

print(f"Popularity Baseline Recall@100: {baseline_recall:.4f}")
print(f"ALS+FAISS Recall@100 (corrected fix): {recall_score:.4f}")

Popularity Baseline Recall@100: 0.1122
ALS+FAISS Recall@100 (corrected fix): 0.0375


In [ ]:
def recall_at_k_positive_only(test_df, k=100, sample_users=2000, use_baseline=False):
    """
    Same as before, but ground truth is now ONLY movies the user genuinely liked (3.5+)
    — matching what our positive-only ALS was actually trained to predict.
    """
    test_users = test_df['userId'].unique()

    if use_baseline:
        eligible_users = test_users
    else:
        eligible_users = [u for u in test_users if u in user_to_idx]

    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    baseline_candidates = set(top_popular_movies[:k])
    hits = 0
    total = 0

    for user_id in sampled_users:
        # KEY CHANGE: only count movies they actually LIKED (3.5+) as ground truth
        actual_liked = set(test_df[(test_df['userId'] == user_id) & (test_df['rating'] >= 3.5)]['movieId'].tolist())

        if len(actual_liked) == 0:
            continue

        candidates = baseline_candidates if use_baseline else set(get_candidates(user_id, k=k))

        hits += len(actual_liked & candidates)
        total += len(actual_liked)

    return hits / total if total > 0 else 0


recall_als_positive = recall_at_k_positive_only(test, k=100, sample_users=2000, use_baseline=False)
recall_baseline_positive = recall_at_k_positive_only(test, k=100, sample_users=2000, use_baseline=True)

print(f"Popularity Baseline Recall@100 (positive-only ground truth): {recall_baseline_positive:.4f}")
print(f"ALS+FAISS Recall@100 (positive-only ground truth): {recall_als_positive:.4f}")

Popularity Baseline Recall@100 (positive-only ground truth): 0.1486
ALS+FAISS Recall@100 (positive-only ground truth): 0.0538


In [ ]:
from implicit.als import AlternatingLeastSquares

results = []

# Test a few different configurations
configs = [
    {"factors": 20, "iterations": 15, "regularization": 0.01},
    {"factors": 100, "iterations": 15, "regularization": 0.01},
    {"factors": 50, "iterations": 30, "regularization": 0.01},
    {"factors": 100, "iterations": 30, "regularization": 0.1},
]

for cfg in configs:
    print(f"Testing: {cfg}")
    test_model = AlternatingLeastSquares(**cfg)
    test_model.fit(user_item_matrix)

    # Temporarily swap in this model's item factors for FAISS
    test_item_vectors = test_model.item_factors.astype('float32')
    test_index = faiss.IndexFlatIP(test_item_vectors.shape[1])
    test_index.add(test_item_vectors)

    # Quick recall check using this config (reusing get_candidates logic inline, smaller sample for speed)
    global model, index
    model, index = test_model, test_index  # temporarily override globals used by get_candidates

    recall = recall_at_k_positive_only(test, k=100, sample_users=500, use_baseline=False)
    print(f"  Recall@100: {recall:.4f}\n")
    results.append({**cfg, "recall": recall})

import pandas as pd
print(pd.DataFrame(results))

Testing: {'factors': 20, 'iterations': 15, 'regularization': 0.01}


  0%|          | 0/15 [00:00<?, ?it/s]

  Recall@100: 0.0618

Testing: {'factors': 100, 'iterations': 15, 'regularization': 0.01}


  0%|          | 0/15 [00:00<?, ?it/s]

  Recall@100: 0.0496

Testing: {'factors': 50, 'iterations': 30, 'regularization': 0.01}


  0%|          | 0/30 [00:00<?, ?it/s]

  Recall@100: 0.0648

Testing: {'factors': 100, 'iterations': 30, 'regularization': 0.1}


  0%|          | 0/30 [00:00<?, ?it/s]

  Recall@100: 0.0455

   factors  iterations  regularization    recall
0       20          15            0.01  0.061838
1      100          15            0.01  0.049597
2       50          30            0.01  0.064756
3      100          30            0.10  0.045533


In [ ]:
# Standard implicit-feedback confidence transformation, per the original ALS paper this library implements
alpha = 40
positive_train = positive_train.copy()
positive_train['confidence'] = 1 + alpha * (positive_train['rating'] - 3.5)  # scale relative to our 3.5 threshold

train_user_idx = positive_train['userId'].map(user_to_idx)
train_movie_idx = positive_train['movieId'].map(movie_to_idx)

user_item_matrix = sparse.csr_matrix(
    (positive_train['confidence'], (train_user_idx, train_movie_idx)),
    shape=(len(user_ids), len(movie_ids))
)

print("Confidence value range:", positive_train['confidence'].min(), "-", positive_train['confidence'].max())
print("Non-zero entries:", user_item_matrix.nnz)

Confidence value range: 1.0 - 61.0
Non-zero entries: 12306450


In [ ]:
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=15
)
model.fit(user_item_matrix)

print("Model retrained with confidence-weighted matrix.")
print("User factors shape:", model.user_factors.shape)
print("Item factors shape:", model.item_factors.shape)

item_vectors = model.item_factors.astype('float32')
dimension = item_vectors.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(item_vectors)

print("FAISS index rebuilt.")
print("Number of vectors indexed:", index.ntotal)

  0%|          | 0/15 [00:00<?, ?it/s]

Model retrained with confidence-weighted matrix.
User factors shape: (137883, 50)
Item factors shape: (34461, 50)
FAISS index rebuilt.
Number of vectors indexed: 34461


In [ ]:
recall_score = recall_at_k_positive_only(test, k=100, sample_users=2000, use_baseline=False)
baseline_recall = recall_at_k_positive_only(test, k=100, sample_users=2000, use_baseline=True)

print(f"Popularity Baseline Recall@100: {baseline_recall:.4f}")
print(f"ALS+FAISS Recall@100 (confidence-weighted): {recall_score:.4f}")

Popularity Baseline Recall@100: 0.1490
ALS+FAISS Recall@100 (confidence-weighted): 0.0627


In [ ]:
alpha_values = [5, 10, 40, 100]
alpha_results = []

for a in alpha_values:
    temp_train = positive_train.copy()
    temp_train['confidence'] = 1 + a * (temp_train['rating'] - 3.5)

    t_user_idx = temp_train['userId'].map(user_to_idx)
    t_movie_idx = temp_train['movieId'].map(movie_to_idx)

    temp_matrix = sparse.csr_matrix(
        (temp_train['confidence'], (t_user_idx, t_movie_idx)),
        shape=(len(user_ids), len(movie_ids))
    )

    temp_model = AlternatingLeastSquares(factors=50, regularization=0.01, iterations=15)
    temp_model.fit(temp_matrix)

    temp_vectors = temp_model.item_factors.astype('float32')
    temp_index = faiss.IndexFlatIP(temp_vectors.shape[1])
    temp_index.add(temp_vectors)

    global model, index
    model, index = temp_model, temp_index

    recall = recall_at_k_positive_only(test, k=100, sample_users=500, use_baseline=False)
    print(f"alpha={a}: Recall@100 = {recall:.4f}")
    alpha_results.append({"alpha": a, "recall": recall})

print(pd.DataFrame(alpha_results))

  0%|          | 0/15 [00:00<?, ?it/s]

alpha=5: Recall@100 = 0.0642


  0%|          | 0/15 [00:00<?, ?it/s]

alpha=10: Recall@100 = 0.0469


  0%|          | 0/15 [00:00<?, ?it/s]

alpha=40: Recall@100 = 0.0620


  0%|          | 0/15 [00:00<?, ?it/s]

alpha=100: Recall@100 = 0.0621
   alpha    recall
0      5  0.064235
1     10  0.046947
2     40  0.062017
3    100  0.062077


In [ ]:
import numpy as np
np.random.seed(42)  # fixes randomness so results are reproducible every time this runs

alpha_values = [5, 10, 40, 100]
alpha_results = []

for a in alpha_values:
    temp_train = positive_train.copy()
    temp_train['confidence'] = 1 + a * (temp_train['rating'] - 3.5)

    t_user_idx = temp_train['userId'].map(user_to_idx)
    t_movie_idx = temp_train['movieId'].map(movie_to_idx)

    temp_matrix = sparse.csr_matrix(
        (temp_train['confidence'], (t_user_idx, t_movie_idx)),
        shape=(len(user_ids), len(movie_ids))
    )

    temp_model = AlternatingLeastSquares(factors=50, regularization=0.01, iterations=15)
    temp_model.fit(temp_matrix)

    temp_vectors = temp_model.item_factors.astype('float32')
    temp_index = faiss.IndexFlatIP(temp_vectors.shape[1])
    temp_index.add(temp_vectors)

    global model, index
    model, index = temp_model, temp_index

    recall = recall_at_k_positive_only(test, k=100, sample_users=1000, use_baseline=False)
    print(f"alpha={a}: Recall@100 = {recall:.4f}")
    alpha_results.append({"alpha": a, "recall": recall})

print(pd.DataFrame(alpha_results))

  0%|          | 0/15 [00:00<?, ?it/s]

alpha=5: Recall@100 = 0.0581


  0%|          | 0/15 [00:00<?, ?it/s]

alpha=10: Recall@100 = 0.0603


  0%|          | 0/15 [00:00<?, ?it/s]

alpha=40: Recall@100 = 0.0605


  0%|          | 0/15 [00:00<?, ?it/s]

alpha=100: Recall@100 = 0.0583
   alpha    recall
0      5  0.058090
1     10  0.060328
2     40  0.060457
3    100  0.058283


In [ ]:
final_alpha = 40
positive_train['confidence'] = 1 + final_alpha * (positive_train['rating'] - 3.5)

train_user_idx = positive_train['userId'].map(user_to_idx)
train_movie_idx = positive_train['movieId'].map(movie_to_idx)

user_item_matrix = sparse.csr_matrix(
    (positive_train['confidence'], (train_user_idx, train_movie_idx)),
    shape=(len(user_ids), len(movie_ids))
)

model = AlternatingLeastSquares(factors=50, regularization=0.01, iterations=15)
model.fit(user_item_matrix)

item_vectors = model.item_factors.astype('float32')
index = faiss.IndexFlatIP(item_vectors.shape[1])
index.add(item_vectors)

print("Final retrieval model trained (alpha=40).")

def get_candidates(user_id, k=100, popular_slots=20):
    """
    Returns top-k candidates: mostly from ALS+FAISS personalization,
    with a small guaranteed slice of popular movies mixed in (hybrid approach)
    to hedge against retrieval's known popularity-bias gap.
    Falls back entirely to popularity for cold-start users.
    """
    if user_id not in user_to_idx:
        return top_popular_movies[:k]

    als_k = k - popular_slots
    user_idx = user_to_idx[user_id]
    user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)
    scores, indices = index.search(user_vector, als_k)
    als_candidates = [int(idx_to_movie[i]) for i in indices[0]]

    popular_fill = [m for m in top_popular_movies if m not in als_candidates][:popular_slots]
    return als_candidates + popular_fill


final_recall = recall_at_k_positive_only(test, k=100, sample_users=2000, use_baseline=False)
print(f"Final hybrid retrieval Recall@100: {final_recall:.4f}")

  0%|          | 0/15 [00:00<?, ?it/s]

Final retrieval model trained (alpha=40).
Final hybrid retrieval Recall@100: 0.0566


In [ ]:
# ===== USER FEATURES =====
# Aggregate stats about each user's rating behavior, computed ONLY from training data
user_features = train.groupby('userId').agg(
    user_rating_count=('rating', 'count'),
    user_avg_rating=('rating', 'mean'),
    user_rating_std=('rating', 'std')
).reset_index()
user_features['user_rating_std'] = user_features['user_rating_std'].fillna(0)  # users with 1 rating have no std

# ===== MOVIE FEATURES =====
# Aggregate stats about each movie, computed ONLY from training data
movie_features = train.groupby('movieId').agg(
    movie_rating_count=('rating', 'count'),
    movie_avg_rating=('rating', 'mean'),
    movie_rating_std=('rating', 'std')
).reset_index()
movie_features['movie_rating_std'] = movie_features['movie_rating_std'].fillna(0)

# Merge in genre info from movies.csv
movie_features = movie_features.merge(movies[['movieId', 'genres', 'title']], on='movieId', how='left')

print("User features shape:", user_features.shape)
print(user_features.head())
print("\nMovie features shape:", movie_features.shape)
print(movie_features.head())

User features shape: (137883, 4)
   userId  user_rating_count  user_avg_rating  user_rating_std
0       1                 70         3.814286         1.004235
1       2                184         3.630435         1.457728
2       3                428         3.746495         0.602968
3       5                101         3.752475         0.931729
4       6                 26         4.153846         0.967153

Movie features shape: (34461, 6)
   movieId  movie_rating_count  movie_avg_rating  movie_rating_std  \
0        1               49853          3.896074          0.911796   
1        2               21196          3.219640          0.957254   
2        3               11514          3.148602          1.004474   
3        4                2479          2.865268          1.100205   
4        5               11122          3.061500          0.988784   

                                        genres  \
0  Adventure|Animation|Children|Comedy|Fantasy   
1                   Adventure|Chil

In [ ]:
def get_als_score(user_id, movie_id):
    """
    Computes the raw ALS compatibility score between a user and movie —
    this is literally the dot product FAISS uses internally to rank candidates.
    """
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return 0.0  # cold-start case: no ALS score available

    user_vec = model.user_factors[user_to_idx[user_id]]
    movie_vec = model.item_factors[movie_to_idx[movie_id]]
    return float(np.dot(user_vec, movie_vec))


# Quick test
test_user = train['userId'].iloc[0]
test_movie = train['movieId'].iloc[0]
print(f"ALS score for user {test_user}, movie {test_movie}: {get_als_score(test_user, test_movie):.4f}")

# Test cold-start case too
print(f"ALS score for unknown user/movie: {get_als_score(999999999, 999999999):.4f}")

ALS score for user 2262, movie 21: 0.5817
ALS score for unknown user/movie: 0.0000


In [ ]:
def build_ranking_features(user_id, candidate_movie_ids):
    """
    For a given user and their list of candidate movies (from retrieval),
    builds a full feature row for each candidate — this is what LightGBM will see.
    """
    rows = []

    # Look up this user's stats once (same for every candidate)
    user_row = user_features[user_features['userId'] == user_id]
    if len(user_row) == 0:
        # Cold-start user: use dataset-wide averages as a reasonable default
        u_count, u_avg, u_std = 0, train['rating'].mean(), 0
    else:
        u_count = user_row['user_rating_count'].values[0]
        u_avg = user_row['user_avg_rating'].values[0]
        u_std = user_row['user_rating_std'].values[0]

    for movie_id in candidate_movie_ids:
        movie_row = movie_features[movie_features['movieId'] == movie_id]
        if len(movie_row) == 0:
            m_count, m_avg, m_std, genres = 0, train['rating'].mean(), 0, ""
        else:
            m_count = movie_row['movie_rating_count'].values[0]
            m_avg = movie_row['movie_avg_rating'].values[0]
            m_std = movie_row['movie_rating_std'].values[0]
            genres = movie_row['genres'].values[0]

        als_score = get_als_score(user_id, movie_id)

        rows.append({
            'userId': user_id,
            'movieId': movie_id,
            'als_score': als_score,
            'user_rating_count': u_count,
            'user_avg_rating': u_avg,
            'user_rating_std': u_std,
            'movie_rating_count': m_count,
            'movie_avg_rating': m_avg,
            'movie_rating_std': m_std,
            'genres': genres,
        })

    return pd.DataFrame(rows)


# Quick test: build features for one user's candidates
test_user = train['userId'].iloc[0]
test_candidates = get_candidates(test_user, k=10)  # just 10 for a quick readable test
feature_df = build_ranking_features(test_user, test_candidates)
print(feature_df)

AssertionError: 

In [ ]:
def get_candidates(user_id, k=100, popular_slots=20):
    if user_id not in user_to_idx:
        return top_popular_movies[:k]

    # Safety: don't let popular_slots exceed k itself, and never let als_k go below 1
    actual_popular_slots = min(popular_slots, max(k - 1, 0))
    als_k = max(k - actual_popular_slots, 1)

    user_idx = user_to_idx[user_id]
    user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)
    scores, indices = index.search(user_vector, als_k)
    als_candidates = [int(idx_to_movie[i]) for i in indices[0]]

    popular_fill = [m for m in top_popular_movies if m not in als_candidates][:actual_popular_slots]
    return als_candidates + popular_fill

In [ ]:
test_user = train['userId'].iloc[0]
test_candidates = get_candidates(test_user, k=10)
feature_df = build_ranking_features(test_user, test_candidates)
print(feature_df)

NameError: name 'build_ranking_features' is not defined

In [ ]:
# Step 1: Break out movies into (movieId, genre) pairs — one row per genre per movie
movies_genres = movies[['movieId', 'genres']].copy()
movies_genres = movies_genres.assign(genre=movies_genres['genres'].str.split('|')).explode('genre')
movies_genres = movies_genres[movies_genres['genre'] != '(no genres listed)']

# Step 2: Merge with POSITIVE training ratings to see which genres each user actually likes
user_genre_ratings = positive_train.merge(movies_genres[['movieId', 'genre']], on='movieId', how='left')

# Step 3: Compute each user's average rating per genre — their "genre affinity"
user_genre_affinity = user_genre_ratings.groupby(['userId', 'genre'])['rating'].mean().reset_index()
user_genre_affinity.columns = ['userId', 'genre', 'genre_affinity']

print("User-genre affinity table shape:", user_genre_affinity.shape)
print(user_genre_affinity.head(10))


def get_genre_match_score(user_id, movie_genres_str):
    """
    For a candidate movie's genre string (e.g. "Comedy|Crime|Drama"),
    returns the average of the user's affinity scores across those genres.
    Higher score = this movie's genres align with what the user rates highly.
    """
    if pd.isna(movie_genres_str) or movie_genres_str == '':
        return 0.0

    genres = movie_genres_str.split('|')
    user_scores = user_genre_affinity[
        (user_genre_affinity['userId'] == user_id) & (user_genre_affinity['genre'].isin(genres))
    ]

    if len(user_scores) == 0:
        return 0.0  # user has no history with any of these genres

    return float(user_scores['genre_affinity'].mean())


# Quick test
print(f"\nGenre match for user {test_user}, 'Action|Sci-Fi|Thriller': {get_genre_match_score(test_user, 'Action|Sci-Fi|Thriller'):.4f}")

User-genre affinity table shape: (2088780, 3)
   userId      genre  genre_affinity
0       1     Action        4.500000
1       1  Adventure        4.125000
2       1  Animation        4.000000
3       1   Children        3.833333
4       1     Comedy        4.277778
5       1      Crime        4.500000
6       1      Drama        4.304878
7       1    Fantasy        3.875000
8       1  Film-Noir        3.500000
9       1     Horror        3.500000

Genre match for user 2262, 'Action|Sci-Fi|Thriller': 4.3658


In [ ]:
def build_ranking_features(user_id, candidate_movie_ids):
    """
    For a given user and their list of candidate movies (from retrieval),
    builds a full feature row for each candidate — now including genre match.
    """
    rows = []

    user_row = user_features[user_features['userId'] == user_id]
    if len(user_row) == 0:
        u_count, u_avg, u_std = 0, train['rating'].mean(), 0
    else:
        u_count = user_row['user_rating_count'].values[0]
        u_avg = user_row['user_avg_rating'].values[0]
        u_std = user_row['user_rating_std'].values[0]

    for movie_id in candidate_movie_ids:
        movie_row = movie_features[movie_features['movieId'] == movie_id]
        if len(movie_row) == 0:
            m_count, m_avg, m_std, genres = 0, train['rating'].mean(), 0, ""
        else:
            m_count = movie_row['movie_rating_count'].values[0]
            m_avg = movie_row['movie_avg_rating'].values[0]
            m_std = movie_row['movie_rating_std'].values[0]
            genres = movie_row['genres'].values[0]

        als_score = get_als_score(user_id, movie_id)
        genre_match = get_genre_match_score(user_id, genres)

        rows.append({
            'userId': user_id,
            'movieId': movie_id,
            'als_score': als_score,
            'user_rating_count': u_count,
            'user_avg_rating': u_avg,
            'user_rating_std': u_std,
            'movie_rating_count': m_count,
            'movie_avg_rating': m_avg,
            'movie_rating_std': m_std,
            'genre_match': genre_match,
        })

    return pd.DataFrame(rows)


print("build_ranking_features updated with genre_match.")

build_ranking_features updated with genre_match.


In [ ]:
def build_training_data(sample_users=5000, k=100, seed=42):
    """
    Builds the full labeled dataset for LightGBM:
    - For each sampled user, get their retrieval candidates
    - Build features for each candidate
    - Label = 1 if the user actually rated that movie positively (3.5+) in val, else 0
    """
    np.random.seed(seed)

    # Only users who exist in both train (for ALS) and val (to have something to predict)
    val_users = val['userId'].unique()
    eligible_users = [u for u in val_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    all_rows = []

    for i, user_id in enumerate(sampled_users):
        if i % 500 == 0:
            print(f"Processing user {i}/{len(sampled_users)}...")

        # What did this user actually like in the validation period? (our ground truth for labeling)
        actual_liked = set(val[(val['userId'] == user_id) & (val['rating'] >= 3.5)]['movieId'].tolist())

        # Get this user's candidates from retrieval
        candidates = get_candidates(user_id, k=k)

        # Build features for these candidates
        feature_df = build_ranking_features(user_id, candidates)

        # Label: 1 if this candidate is something the user actually liked in val, else 0
        feature_df['label'] = feature_df['movieId'].apply(lambda m: 1 if m in actual_liked else 0)

        all_rows.append(feature_df)

    return pd.concat(all_rows, ignore_index=True)


training_data = build_training_data(sample_users=5000, k=100)
print("\nTraining data shape:", training_data.shape)
print("Positive labels:", training_data['label'].sum())
print("Negative labels:", (training_data['label'] == 0).sum())
print(training_data.head())

Processing user 0/5000...


KeyboardInterrupt: 

In [ ]:
# Convert to fast dictionary lookups instead of slow dataframe filtering
user_features_dict = user_features.set_index('userId').to_dict('index')
movie_features_dict = movie_features.set_index('movieId').to_dict('index')
genre_affinity_dict = user_genre_affinity.groupby('userId').apply(
    lambda x: dict(zip(x['genre'], x['genre_affinity']))
).to_dict()

print("Fast lookup dictionaries built.")

/tmp/ipykernel_2743/3913058842.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  genre_affinity_dict = user_genre_affinity.groupby('userId').apply(


Fast lookup dictionaries built.


In [ ]:
def get_genre_match_score_fast(user_id, movie_genres_str):
    if pd.isna(movie_genres_str) or movie_genres_str == '' or user_id not in genre_affinity_dict:
        return 0.0
    genres = movie_genres_str.split('|')
    user_genre_scores = genre_affinity_dict[user_id]
    scores = [user_genre_scores[g] for g in genres if g in user_genre_scores]
    return float(np.mean(scores)) if scores else 0.0


def build_ranking_features_fast(user_id, candidate_movie_ids):
    rows = []

    if user_id in user_features_dict:
        u = user_features_dict[user_id]
        u_count, u_avg, u_std = u['user_rating_count'], u['user_avg_rating'], u['user_rating_std']
    else:
        u_count, u_avg, u_std = 0, train['rating'].mean(), 0

    for movie_id in candidate_movie_ids:
        if movie_id in movie_features_dict:
            m = movie_features_dict[movie_id]
            m_count, m_avg, m_std, genres = m['movie_rating_count'], m['movie_avg_rating'], m['movie_rating_std'], m['genres']
        else:
            m_count, m_avg, m_std, genres = 0, train['rating'].mean(), 0, ""

        als_score = get_als_score(user_id, movie_id)
        genre_match = get_genre_match_score_fast(user_id, genres)

        rows.append({
            'userId': user_id, 'movieId': movie_id, 'als_score': als_score,
            'user_rating_count': u_count, 'user_avg_rating': u_avg, 'user_rating_std': u_std,
            'movie_rating_count': m_count, 'movie_avg_rating': m_avg, 'movie_rating_std': m_std,
            'genre_match': genre_match,
        })

    return pd.DataFrame(rows)


print("Fast functions ready.")

Fast functions ready.


In [ ]:
def build_training_data(sample_users=5000, k=100, seed=42):
    np.random.seed(seed)

    val_users = val['userId'].unique()
    eligible_users = [u for u in val_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    all_rows = []

    for i, user_id in enumerate(sampled_users):
        if i % 500 == 0:
            print(f"Processing user {i}/{len(sampled_users)}...")

        actual_liked = set(val[(val['userId'] == user_id) & (val['rating'] >= 3.5)]['movieId'].tolist())
        candidates = get_candidates(user_id, k=k)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['label'] = feature_df['movieId'].apply(lambda m: 1 if m in actual_liked else 0)
        all_rows.append(feature_df)

    return pd.concat(all_rows, ignore_index=True)


training_data = build_training_data(sample_users=5000, k=100)
print("\nTraining data shape:", training_data.shape)
print("Positive labels:", training_data['label'].sum())
print("Negative labels:", (training_data['label'] == 0).sum())
print(training_data.head())

Processing user 0/5000...
Processing user 500/5000...
Processing user 1000/5000...
Processing user 1500/5000...
Processing user 2000/5000...
Processing user 2500/5000...
Processing user 3000/5000...
Processing user 3500/5000...
Processing user 4000/5000...
Processing user 4500/5000...

Training data shape: (500000, 11)
Positive labels: 20087
Negative labels: 479913
   userId  movieId  als_score  user_rating_count  user_avg_rating  \
0  110401    34405   1.155039                179         3.921788   
1  110401     1246   1.133536                179         3.921788   
2  110401     1197   1.125613                179         3.921788   
3  110401    40815   1.119194                179         3.921788   
4  110401    66934   1.117978                179         3.921788   

   user_rating_std  movie_rating_count  movie_avg_rating  movie_rating_std  \
0         0.760341                8864          3.992103          0.882780   
1         0.760341               20115          3.903555     

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit

# Sort by userId so all of one user's candidates are grouped together — required for ranking mode
training_data = training_data.sort_values('userId').reset_index(drop=True)

# Split by USER (not by row) so a user's candidates don't get split across train/val —
# otherwise LightGBM would be learning to rank incomplete groups, which breaks ranking logic
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data, groups=training_data['userId']))

lgb_train_df = training_data.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data.iloc[val_idx].sort_values('userId').reset_index(drop=True)

# Feature columns LightGBM will actually use (excluding IDs and the label itself)
feature_cols = ['als_score', 'user_rating_count', 'user_avg_rating', 'user_rating_std',
                 'movie_rating_count', 'movie_avg_rating', 'movie_rating_std', 'genre_match']

# Group sizes: how many candidate rows belong to each user, in order — LightGBM needs this to know
# where one user's list ends and the next user's list begins
lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

print("LightGBM training rows:", len(lgb_train_df), "| users:", len(lgb_train_groups))
print("LightGBM validation rows:", len(lgb_val_df), "| users:", len(lgb_val_groups))

LightGBM training rows: 400000 | users: 4000
LightGBM validation rows: 100000 | users: 1000


In [ ]:
train_dataset = lgb.Dataset(
    lgb_train_df[feature_cols],
    label=lgb_train_df['label'],
    group=lgb_train_groups
)

val_dataset = lgb.Dataset(
    lgb_val_df[feature_cols],
    label=lgb_val_df['label'],
    group=lgb_val_groups,
    reference=train_dataset
)

params = {
    'objective': 'lambdarank',   # ranking-specific objective, not plain classification
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

ranker = lgb.train(
    params,
    train_dataset,
    num_boost_round=200,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(period=20)]
)

print("LightGBM ranker trained.")

Training until validation scores don't improve for 20 rounds
[20]	valid_0's ndcg@10: 0.40913
[40]	valid_0's ndcg@10: 0.407806
Early stopping, best iteration is:
[34]	valid_0's ndcg@10: 0.409334
LightGBM ranker trained.


In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': ranker.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print(importance)

              feature   importance
4  movie_rating_count  2624.410342
5    movie_avg_rating  2514.666527
0           als_score  2345.271054
1   user_rating_count  2068.314364
6    movie_rating_std  1947.076984
7         genre_match  1765.163299
2     user_avg_rating  1420.880293
3     user_rating_std  1266.341161


In [ ]:
def evaluate_ranker_on_test(sample_users=2000, k_retrieve=100, k_final=10, seed=42):
    """
    The real, final test: for users in the TEST set (never touched until now),
    retrieve candidates, rank them with our trained LightGBM model,
    and check Precision@10 / NDCG@10 against what they actually liked.
    """
    np.random.seed(seed)

    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    precisions = []
    ndcgs = []

    for i, user_id in enumerate(sampled_users):
        if i % 500 == 0:
            print(f"Evaluating user {i}/{len(sampled_users)}...")

        actual_liked = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        if len(actual_liked) == 0:
            continue

        # Retrieval stage
        candidates = get_candidates(user_id, k=k_retrieve)

        # Build features for ranking
        feature_df = build_ranking_features_fast(user_id, candidates)

        # Ranking stage: predict scores, sort to get final top-k_final
        feature_df['score'] = ranker.predict(feature_df[feature_cols])
        top_final = feature_df.sort_values('score', ascending=False).head(k_final)

        recommended_movies = top_final['movieId'].tolist()

        # Precision@10: of our final 10, how many did they actually like?
        hits = len(set(recommended_movies) & actual_liked)
        precisions.append(hits / k_final)

        # NDCG@10: rewards getting the right ones near the TOP of the 10
        relevance = [1 if m in actual_liked else 0 for m in recommended_movies]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        ideal_relevance = sorted(relevance, reverse=True)
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(ideal_relevance))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(precisions), np.mean(ndcgs)


final_precision, final_ndcg = evaluate_ranker_on_test(sample_users=2000)
print(f"\nFinal Precision@10 (test set): {final_precision:.4f}")
print(f"Final NDCG@10 (test set): {final_ndcg:.4f}")

Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

Final Precision@10 (test set): 0.0360
Final NDCG@10 (test set): 0.1154


In [ ]:
def evaluate_baseline_on_test(sample_users=2000, k_final=10, seed=42):
    """
    Same evaluation, but using the popularity baseline's top-10 for every user —
    our comparison point, exactly like we did for retrieval.
    """
    np.random.seed(seed)

    test_users = test['userId'].unique()
    sampled_users = np.random.choice(test_users, size=min(sample_users, len(test_users)), replace=False)

    baseline_top10 = top_popular_movies[:k_final]

    precisions = []
    ndcgs = []

    for user_id in sampled_users:
        actual_liked = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        if len(actual_liked) == 0:
            continue

        hits = len(set(baseline_top10) & actual_liked)
        precisions.append(hits / k_final)

        relevance = [1 if m in actual_liked else 0 for m in baseline_top10]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        ideal_relevance = sorted(relevance, reverse=True)
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(ideal_relevance))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(precisions), np.mean(ndcgs)


baseline_precision, baseline_ndcg = evaluate_baseline_on_test(sample_users=2000)
print(f"Baseline Precision@10 (test set): {baseline_precision:.4f}")
print(f"Baseline NDCG@10 (test set): {baseline_ndcg:.4f}")
print(f"\nOur system Precision@10: {final_precision:.4f} | NDCG@10: {final_ndcg:.4f}")
print(f"Baseline Precision@10: {baseline_precision:.4f} | NDCG@10: {baseline_ndcg:.4f}")

Baseline Precision@10 (test set): 0.2521
Baseline NDCG@10 (test set): 0.4777

Our system Precision@10: 0.0360 | NDCG@10: 0.1154
Baseline Precision@10: 0.2521 | NDCG@10: 0.4777


In [ ]:
# Diagnostic: inspect exactly what happens for one real test user
diag_user = sampled_users[0] if 'sampled_users' in dir() else test['userId'].iloc[0]
# Use a guaranteed eligible user instead, to be safe:
diag_user = [u for u in test['userId'].unique() if u in user_to_idx][0]

actual_liked = set(test[(test['userId'] == diag_user) & (test['rating'] >= 3.5)]['movieId'].tolist())
candidates = get_candidates(diag_user, k=100)
feature_df = build_ranking_features_fast(diag_user, candidates)
feature_df['score'] = ranker.predict(feature_df[feature_cols])
top10 = feature_df.sort_values('score', ascending=False).head(10)

print(f"User {diag_user} actually liked (in test): {actual_liked}")
print(f"\nOur system's final top 10 (movieId, score, movie_rating_count, als_score):")
print(top10[['movieId', 'score', 'movie_rating_count', 'als_score']])
print(f"\nDoes our top 10 include any actual likes? {set(top10['movieId']) & actual_liked}")
print(f"\nGlobal top 10 popular movies: {top_popular_movies[:10]}")
print(f"Are they in our candidate pool of 100? {[m in candidates for m in top_popular_movies[:10]]}")

User 48988 actually liked (in test): {169984, 74754, 39941, 122886, 4105, 69134, 3599, 60950, 122906, 541, 3101, 158238, 122916, 182823, 103980, 164909, 122926, 92204, 94780, 88125, 116797, 58432, 88129, 7748, 180297, 587, 589, 185425, 192593, 197203, 6754, 6757, 3176, 4713, 179819, 3702, 183421, 2174, 46723, 167564, 27790, 65682, 67734, 168612, 127140, 25770, 2732, 38061, 96432, 5810, 6323, 174261, 155830, 25788, 121545, 55498, 187593, 187595, 2762, 184017, 7377, 1240, 31963, 32988, 1261, 42738, 176371, 171763, 31991, 86781, 197889, 192777, 1291, 84242, 2840, 183073, 162082, 176419, 169764, 176933, 3365, 169256, 133419, 4914, 8507, 168252, 2877, 179005, 177469, 6987, 81229, 61262, 33615, 1387, 2926, 2420, 159093, 141688, 103801, 7043, 168328, 168330, 105355, 193944, 7065, 1438, 174503, 193449, 7090, 170937, 5569, 156609, 58306, 193477, 1997, 185295, 3535, 76763, 7135, 7136, 34271, 190945, 191459, 176101, 174055, 135143, 7151}

Our system's final top 10 (movieId, score, movie_rating_co

In [ ]:
# Check: does label rate (fraction of label=1) differ by movie popularity level?
training_data_with_pop = training_data.merge(
    movie_features[['movieId', 'movie_rating_count']], on='movieId', how='left', suffixes=('', '_check')
)

# Bucket movies by popularity
training_data['popularity_bucket'] = pd.qcut(training_data['movie_rating_count'], q=5, duplicates='drop')
label_rate_by_popularity = training_data.groupby('popularity_bucket')['label'].mean()

print("Label rate (fraction that are actual likes) by movie popularity bucket:")
print(label_rate_by_popularity)

Label rate (fraction that are actual likes) by movie popularity bucket:
popularity_bucket
(87.999, 5610.0]      0.037618
(5610.0, 12012.0]     0.039195
(12012.0, 24272.0]    0.043845
(24272.0, 42985.2]    0.043965
(42985.2, 67782.0]    0.036270
Name: label, dtype: float64


/tmp/ipykernel_2743/1798958914.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  label_rate_by_popularity = training_data.groupby('popularity_bucket')['label'].mean()


In [ ]:
# Check: how many times does each movie appear in training_data, and what's its label rate specifically?
movie_appearance_stats = training_data.groupby('movieId').agg(
    times_appeared=('label', 'count'),
    label_rate=('label', 'mean')
).reset_index()

# Our specific popularity fallback movies (top 20)
fallback_movies = set(top_popular_movies[:20])
movie_appearance_stats['is_fallback'] = movie_appearance_stats['movieId'].isin(fallback_movies)

print("Fallback movies (appear in almost every user's candidate list):")
print(movie_appearance_stats[movie_appearance_stats['is_fallback']].sort_values('times_appeared', ascending=False))

print("\nAverage label rate — fallback movies vs non-fallback movies:")
print(movie_appearance_stats.groupby('is_fallback')['label_rate'].mean())

print("\nAverage times appeared — fallback vs non-fallback:")
print(movie_appearance_stats.groupby('is_fallback')['times_appeared'].mean())

Fallback movies (appear in almost every user's candidate list):
      movieId  times_appeared  label_rate  is_fallback
0           1            5000      0.0332         True
46         50            5000      0.0428         True
87        110            5000      0.0300         True
99        150            5000      0.0258         True
155       260            5000      0.0400         True
171       296            5000      0.0464         True
184       318            5000      0.0458         True
206       356            5000      0.0416         True
247       457            5000      0.0214         True
254       480            5000      0.0356         True
276       527            5000      0.0322         True
300       589            5000      0.0320         True
303       593            5000      0.0382         True
351       780            5000      0.0256         True
531      1196            5000      0.0382         True
533      1198            5000      0.0310         True
5

In [ ]:
def get_candidates_pure_als(user_id, k=100):
    """
    Pure ALS+FAISS candidates, NO forced popularity injection.
    Used ONLY for building training data, to avoid the duplication bug
    we just diagnosed. The hybrid get_candidates() stays as-is for real serving.
    """
    if user_id not in user_to_idx:
        return top_popular_movies[:k]

    user_idx = user_to_idx[user_id]
    user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)
    scores, indices = index.search(user_vector, k)
    candidates = [int(idx_to_movie[i]) for i in indices[0]]

    return candidates


# Quick test to confirm it works
test_candidates_pure = get_candidates_pure_als(test['userId'].iloc[0], k=10)
print("Pure ALS candidates (no forced popularity):", test_candidates_pure)

Pure ALS candidates (no forced popularity): [356, 296, 318, 593, 480, 260, 2571, 110, 527, 589]


In [ ]:
def build_training_data_v2(sample_users=5000, k=100, seed=42):
    np.random.seed(seed)

    val_users = val['userId'].unique()
    eligible_users = [u for u in val_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    all_rows = []

    for i, user_id in enumerate(sampled_users):
        if i % 500 == 0:
            print(f"Processing user {i}/{len(sampled_users)}...")

        actual_liked = set(val[(val['userId'] == user_id) & (val['rating'] >= 3.5)]['movieId'].tolist())

        # KEY CHANGE: pure ALS candidates, not the hybrid version
        candidates = get_candidates_pure_als(user_id, k=k)

        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['label'] = feature_df['movieId'].apply(lambda m: 1 if m in actual_liked else 0)
        all_rows.append(feature_df)

    return pd.concat(all_rows, ignore_index=True)


training_data_v2 = build_training_data_v2(sample_users=5000, k=100)
print("\nTraining data v2 shape:", training_data_v2.shape)
print("Positive labels:", training_data_v2['label'].sum())
print("Negative labels:", (training_data_v2['label'] == 0).sum())

# Verify the fix: check if movie appearance counts are now more evenly distributed
appearance_check = training_data_v2.groupby('movieId').size()
print("\nMax times any single movie appears now:", appearance_check.max())
print("(Previously, fallback movies appeared exactly 5000 times each)")

Processing user 0/5000...
Processing user 500/5000...
Processing user 1000/5000...
Processing user 1500/5000...
Processing user 2000/5000...
Processing user 2500/5000...
Processing user 3000/5000...
Processing user 3500/5000...
Processing user 4000/5000...
Processing user 4500/5000...

Training data v2 shape: (500000, 11)
Positive labels: 20957
Negative labels: 479043

Max times any single movie appears now: 3024
(Previously, fallback movies appeared exactly 5000 times each)


In [ ]:
# Sort and split by user, same as before, but using training_data_v2
training_data_v2 = training_data_v2.sort_values('userId').reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data_v2, groups=training_data_v2['userId']))

lgb_train_df = training_data_v2.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data_v2.iloc[val_idx].sort_values('userId').reset_index(drop=True)

lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

train_dataset = lgb.Dataset(
    lgb_train_df[feature_cols],
    label=lgb_train_df['label'],
    group=lgb_train_groups
)

val_dataset = lgb.Dataset(
    lgb_val_df[feature_cols],
    label=lgb_val_df['label'],
    group=lgb_val_groups,
    reference=train_dataset
)

params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

ranker = lgb.train(
    params,
    train_dataset,
    num_boost_round=200,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(period=20)]
)

print("LightGBM ranker retrained on corrected data.")

# Check feature importance again
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': ranker.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
print(importance)

Training until validation scores don't improve for 20 rounds
[20]	valid_0's ndcg@10: 0.407094
Early stopping, best iteration is:
[19]	valid_0's ndcg@10: 0.407982
LightGBM ranker retrained on corrected data.
              feature   importance
4  movie_rating_count  2283.023808
6    movie_rating_std  2162.893879
5    movie_avg_rating  1977.780530
1   user_rating_count  1352.499148
0           als_score  1265.937394
7         genre_match   875.595261
2     user_avg_rating   784.937679
3     user_rating_std   442.326802


In [ ]:
final_precision_v2, final_ndcg_v2 = evaluate_ranker_on_test(sample_users=2000)
print(f"\nFinal Precision@10 (test set, corrected model): {final_precision_v2:.4f}")
print(f"Final NDCG@10 (test set, corrected model): {final_ndcg_v2:.4f}")

print(f"\n--- Comparison ---")
print(f"Baseline:              Precision@10: {baseline_precision:.4f} | NDCG@10: {baseline_ndcg:.4f}")
print(f"Our system (buggy):    Precision@10: {final_precision:.4f} | NDCG@10: {final_ndcg:.4f}")
print(f"Our system (corrected): Precision@10: {final_precision_v2:.4f} | NDCG@10: {final_ndcg_v2:.4f}")

Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

Final Precision@10 (test set, corrected model): 0.0347
Final NDCG@10 (test set, corrected model): 0.1127

--- Comparison ---
Baseline:              Precision@10: 0.2521 | NDCG@10: 0.4777
Our system (buggy):    Precision@10: 0.0360 | NDCG@10: 0.1154
Our system (corrected): Precision@10: 0.0347 | NDCG@10: 0.1127


In [ ]:
# Check: how many of the movies test users actually liked are structurally unreachable
# (never existed in our training-time movie universe at all)
test_positive_movies_v2 = set(test[test['rating'] >= 3.5]['movieId'].unique())
all_training_movies = set(train['movieId'].unique())

never_in_training = test_positive_movies_v2 - all_training_movies
print(f"Test-positive movies never seen in training: {len(never_in_training)}")
print(f"Total test-positive movies: {len(test_positive_movies_v2)}")
print(f"Percent structurally unreachable: {len(never_in_training) / len(test_positive_movies_v2) * 100:.1f}%")

# Check specifically for our diagnostic user
diag_liked_unreachable = actual_liked - all_training_movies
print(f"\nFor our diagnostic user: {len(diag_liked_unreachable)} out of {len(actual_liked)} actual likes were never in training")
print(f"Unreachable movie IDs: {diag_liked_unreachable}")

Test-positive movies never seen in training: 14075
Total test-positive movies: 36844
Percent structurally unreachable: 38.2%

For our diagnostic user: 41 out of 127 actual likes were never in training
Unreachable movie IDs: {169984, 197889, 168328, 192777, 168330, 167564, 193944, 183073, 162082, 176419, 168612, 169764, 176933, 182823, 169256, 174503, 193449, 164909, 174261, 155830, 170937, 168252, 179005, 177469, 193477, 180297, 187593, 187595, 185295, 185425, 192593, 197203, 184017, 190945, 191459, 176101, 174055, 179819, 176371, 171763, 183421}


In [ ]:
def evaluate_ranker_on_test_reachable_only(sample_users=2000, k_retrieve=100, k_final=10, seed=42):
    np.random.seed(seed)

    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    precisions = []
    ndcgs = []

    for i, user_id in enumerate(sampled_users):
        if i % 500 == 0:
            print(f"Evaluating user {i}/{len(sampled_users)}...")

        # KEY CHANGE: only count actual likes that were structurally reachable (existed in training)
        actual_liked_raw = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        actual_liked = actual_liked_raw & all_training_movies

        if len(actual_liked) == 0:
            continue

        candidates = get_candidates(user_id, k=k_retrieve)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['score'] = ranker.predict(feature_df[feature_cols])
        top_final = feature_df.sort_values('score', ascending=False).head(k_final)

        recommended_movies = top_final['movieId'].tolist()
        hits = len(set(recommended_movies) & actual_liked)
        precisions.append(hits / k_final)

        relevance = [1 if m in actual_liked else 0 for m in recommended_movies]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        ideal_relevance = sorted(relevance, reverse=True)
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(ideal_relevance))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(precisions), np.mean(ndcgs)


fair_precision, fair_ndcg = evaluate_ranker_on_test_reachable_only(sample_users=2000)
print(f"\nFair Precision@10 (reachable-only ground truth): {fair_precision:.4f}")
print(f"Fair NDCG@10 (reachable-only ground truth): {fair_ndcg:.4f}")
print(f"\nFor comparison — Baseline: Precision@10: {baseline_precision:.4f} | NDCG@10: {baseline_ndcg:.4f}")

Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

Fair Precision@10 (reachable-only ground truth): 0.0359
Fair NDCG@10 (reachable-only ground truth): 0.1164

For comparison — Baseline: Precision@10: 0.2521 | NDCG@10: 0.4777


In [ ]:
def check_precision_ceiling(sample_users=1000, k_retrieve=100, seed=42):
    """
    For each user, count how many of their actual reachable likes
    appear ANYWHERE in the 100 candidates — this is the absolute max
    number of correct items ranking could possibly place in the top 10.
    """
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    relevant_in_pool_counts = []

    for user_id in sampled_users:
        actual_liked_raw = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        actual_liked = actual_liked_raw & all_training_movies
        if len(actual_liked) == 0:
            continue

        candidates = set(get_candidates(user_id, k=k_retrieve))
        relevant_in_pool = len(actual_liked & candidates)
        relevant_in_pool_counts.append(relevant_in_pool)

    relevant_in_pool_counts = np.array(relevant_in_pool_counts)
    print(f"Average relevant items available in each user's 100-candidate pool: {relevant_in_pool_counts.mean():.3f}")
    print(f"Users with ZERO relevant items in their pool: {(relevant_in_pool_counts == 0).mean()*100:.1f}%")
    print(f"Theoretical max Precision@10 (if ranking were PERFECT): {np.minimum(relevant_in_pool_counts, 10).mean() / 10:.4f}")


check_precision_ceiling(sample_users=1000)

Average relevant items available in each user's 100-candidate pool: 2.925
Users with ZERO relevant items in their pool: 39.0%
Theoretical max Precision@10 (if ranking were PERFECT): 0.2230


In [ ]:
# Deep diagnostic: for our diagnostic user, where do the TRUE positives rank
# among all 100 candidates, according to LightGBM's predicted scores?
diag_user = [u for u in test['userId'].unique() if u in user_to_idx][0]
actual_liked_raw = set(test[(test['userId'] == diag_user) & (test['rating'] >= 3.5)]['movieId'].tolist())
actual_liked = actual_liked_raw & all_training_movies

candidates = get_candidates(diag_user, k=100)
feature_df = build_ranking_features_fast(diag_user, candidates)
feature_df['score'] = ranker.predict(feature_df[feature_cols])
feature_df = feature_df.sort_values('score', ascending=False).reset_index(drop=True)
feature_df['rank'] = feature_df.index + 1
feature_df['is_relevant'] = feature_df['movieId'].isin(actual_liked)

print(f"Relevant movies in this user's candidate pool: {feature_df['is_relevant'].sum()}")
print(f"\nWhere did LightGBM rank them (out of 100)?")
print(feature_df[feature_df['is_relevant']][['movieId', 'rank', 'score', 'als_score', 'movie_rating_count']])

print(f"\nScore distribution overall:")
print(feature_df['score'].describe())

Relevant movies in this user's candidate pool: 6

Where did LightGBM rank them (out of 100)?
    movieId  rank     score  als_score  movie_rating_count
1    122886     2  0.147586   1.077897                4435
28    46723    29 -0.022462   1.122466                4599
55     5810    56 -0.064826   1.116849                5431
57     3535    58 -0.065076   1.075925                9100
75     2174    76 -0.114511   1.108620               18590
81      589    82 -0.129792   0.841305               50400

Score distribution overall:
count    100.000000
mean      -0.054966
std        0.086485
min       -0.211088
25%       -0.112777
50%       -0.055000
75%       -0.010799
max        0.446969
Name: score, dtype: float64


In [ ]:
training_data_v3 = build_training_data_v2(sample_users=12000, k=100, seed=42)
print("\nTraining data v3 shape:", training_data_v3.shape)
print("Positive labels:", training_data_v3['label'].sum())
print("Negative labels:", (training_data_v3['label'] == 0).sum())

Processing user 0/5295...
Processing user 500/5295...
Processing user 1000/5295...
Processing user 1500/5295...
Processing user 2000/5295...
Processing user 2500/5295...
Processing user 3000/5295...
Processing user 3500/5295...
Processing user 4000/5295...
Processing user 4500/5295...
Processing user 5000/5295...

Training data v3 shape: (529500, 11)
Positive labels: 22213
Negative labels: 507287


In [ ]:
training_data_v3 = training_data_v3.sort_values('userId').reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data_v3, groups=training_data_v3['userId']))

lgb_train_df = training_data_v3.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data_v3.iloc[val_idx].sort_values('userId').reset_index(drop=True)

lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

train_dataset = lgb.Dataset(lgb_train_df[feature_cols], label=lgb_train_df['label'], group=lgb_train_groups)
val_dataset = lgb.Dataset(lgb_val_df[feature_cols], label=lgb_val_df['label'], group=lgb_val_groups, reference=train_dataset)

params_v2 = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.03,
    'num_leaves': 63,          # more capacity to learn finer distinctions
    'min_data_in_leaf': 20,    # prevent overfitting despite more leaves
    'lambdarank_truncation_level': 10,  # focus learning specifically on top-10 correctness
    'verbose': -1
}

ranker = lgb.train(
    params_v2,
    train_dataset,
    num_boost_round=400,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=20)]
)

print("LightGBM retrained with tuned params.")
importance = pd.DataFrame({'feature': feature_cols, 'importance': ranker.feature_importance(importance_type='gain')}).sort_values('importance', ascending=False)
print(importance)

Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@10: 0.401545
[40]	valid_0's ndcg@10: 0.403813
[60]	valid_0's ndcg@10: 0.405975
[80]	valid_0's ndcg@10: 0.404102
Early stopping, best iteration is:
[52]	valid_0's ndcg@10: 0.40729
LightGBM retrained with tuned params.
              feature   importance
4  movie_rating_count  7729.766793
5    movie_avg_rating  7645.765675
0           als_score  7545.510252
1   user_rating_count  6671.047158
7         genre_match  6218.060265
6    movie_rating_std  6074.294662
3     user_rating_std  5335.338592
2     user_avg_rating  5216.072131


In [ ]:
final_precision_v3, final_ndcg_v3 = evaluate_ranker_on_test(sample_users=2000)
print(f"\nFinal Precision@10 (test set, tuned model): {final_precision_v3:.4f}")
print(f"Final NDCG@10 (test set, tuned model): {final_ndcg_v3:.4f}")

print(f"\n--- Full Comparison ---")
print(f"Baseline:                Precision@10: {baseline_precision:.4f} | NDCG@10: {baseline_ndcg:.4f}")
print(f"Our system (v1, buggy):  Precision@10: {final_precision:.4f} | NDCG@10: {final_ndcg:.4f}")
print(f"Our system (v2, fixed duplication): Precision@10: {final_precision_v2:.4f} | NDCG@10: {final_ndcg_v2:.4f}")
print(f"Our system (v3, tuned):  Precision@10: {final_precision_v3:.4f} | NDCG@10: {final_ndcg_v3:.4f}")

Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

Final Precision@10 (test set, tuned model): 0.0341
Final NDCG@10 (test set, tuned model): 0.1084

--- Full Comparison ---
Baseline:                Precision@10: 0.2521 | NDCG@10: 0.4777
Our system (v1, buggy):  Precision@10: 0.0360 | NDCG@10: 0.1154
Our system (v2, fixed duplication): Precision@10: 0.0347 | NDCG@10: 0.1127
Our system (v3, tuned):  Precision@10: 0.0341 | NDCG@10: 0.1084


In [ ]:
def build_training_data_negsample(sample_users=12000, k=100, neg_ratio=5, seed=42):
    """
    Same as before, but instead of keeping all ~97 negatives per user,
    we keep only a sampled subset (neg_ratio times the number of positives)
    — reducing the overwhelming, noisy negative signal from MNAR bias.
    """
    np.random.seed(seed)

    val_users = val['userId'].unique()
    eligible_users = [u for u in val_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    all_rows = []

    for i, user_id in enumerate(sampled_users):
        if i % 500 == 0:
            print(f"Processing user {i}/{len(sampled_users)}...")

        actual_liked = set(val[(val['userId'] == user_id) & (val['rating'] >= 3.5)]['movieId'].tolist())
        candidates = get_candidates_pure_als(user_id, k=k)

        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['label'] = feature_df['movieId'].apply(lambda m: 1 if m in actual_liked else 0)

        positives = feature_df[feature_df['label'] == 1]
        negatives = feature_df[feature_df['label'] == 0]

        n_pos = len(positives)
        if n_pos == 0:
            continue  # skip users with zero positives — nothing to rank against

        n_neg_to_keep = min(len(negatives), max(n_pos * neg_ratio, 10))
        sampled_negatives = negatives.sample(n=n_neg_to_keep, random_state=seed)

        user_final = pd.concat([positives, sampled_negatives]).sort_values('userId')
        all_rows.append(user_final)

    return pd.concat(all_rows, ignore_index=True)


training_data_neg = build_training_data_negsample(sample_users=12000, k=100, neg_ratio=5)
print("\nTraining data (negative sampled) shape:", training_data_neg.shape)
print("Positive labels:", training_data_neg['label'].sum())
print("Negative labels:", (training_data_neg['label'] == 0).sum())
print("Positive ratio:", training_data_neg['label'].mean())

Processing user 0/5295...
Processing user 500/5295...
Processing user 1000/5295...
Processing user 1500/5295...
Processing user 2000/5295...
Processing user 2500/5295...
Processing user 3000/5295...
Processing user 3500/5295...
Processing user 4000/5295...
Processing user 4500/5295...
Processing user 5000/5295...

Training data (negative sampled) shape: (122478, 11)
Positive labels: 22213
Negative labels: 100265
Positive ratio: 0.18136318359215534


In [ ]:
training_data_neg = training_data_neg.sort_values('userId').reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data_neg, groups=training_data_neg['userId']))

lgb_train_df = training_data_neg.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data_neg.iloc[val_idx].sort_values('userId').reset_index(drop=True)

lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

train_dataset = lgb.Dataset(lgb_train_df[feature_cols], label=lgb_train_df['label'], group=lgb_train_groups)
val_dataset = lgb.Dataset(lgb_val_df[feature_cols], label=lgb_val_df['label'], group=lgb_val_groups, reference=train_dataset)

params_v3 = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_data_in_leaf': 20,
    'lambdarank_truncation_level': 10,
    'verbose': -1
}

ranker = lgb.train(
    params_v3,
    train_dataset,
    num_boost_round=400,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=20)]
)

print("LightGBM retrained on negative-sampled data.")

# Real test - the moment of truth
final_precision_v4, final_ndcg_v4 = evaluate_ranker_on_test(sample_users=2000)
print(f"\nFinal Precision@10 (test set, negative-sampled model): {final_precision_v4:.4f}")
print(f"Final NDCG@10 (test set, negative-sampled model): {final_ndcg_v4:.4f}")

print(f"\n--- Full Comparison ---")
print(f"Baseline:                Precision@10: {baseline_precision:.4f} | NDCG@10: {baseline_ndcg:.4f}")
print(f"Our system (v3, tuned):  Precision@10: {final_precision_v3:.4f} | NDCG@10: {final_ndcg_v3:.4f}")
print(f"Our system (v4, neg-sampled): Precision@10: {final_precision_v4:.4f} | NDCG@10: {final_ndcg_v4:.4f}")

Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@10: 0.407371
[40]	valid_0's ndcg@10: 0.404001
Early stopping, best iteration is:
[20]	valid_0's ndcg@10: 0.407371
LightGBM retrained on negative-sampled data.
Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

Final Precision@10 (test set, negative-sampled model): 0.0369
Final NDCG@10 (test set, negative-sampled model): 0.1173

--- Full Comparison ---
Baseline:                Precision@10: 0.2521 | NDCG@10: 0.4777
Our system (v3, tuned):  Precision@10: 0.0341 | NDCG@10: 0.1084
Our system (v4, neg-sampled): Precision@10: 0.0369 | NDCG@10: 0.1173


In [ ]:
# ===== RECENCY FEATURE =====
# For each movie, find its most recent rating timestamp in the training period,
# then express "how recently active" it is relative to the training window's end
train_max_date = train['datetime'].max()

movie_recency = train.groupby('movieId')['datetime'].max().reset_index()
movie_recency.columns = ['movieId', 'last_rated_date']
movie_recency['days_since_last_rating'] = (train_max_date - movie_recency['last_rated_date']).dt.days

# ===== IMPROVED POPULARITY FEATURES =====
# Log-scale the raw count (compresses the huge range between niche and blockbuster movies)
# and compute percentile rank (where does this movie sit relative to all others?)
movie_features['movie_rating_count_log'] = np.log1p(movie_features['movie_rating_count'])
movie_features['movie_popularity_percentile'] = movie_features['movie_rating_count'].rank(pct=True)

# Merge recency into movie_features
movie_features = movie_features.merge(
    movie_recency[['movieId', 'days_since_last_rating']], on='movieId', how='left'
)
movie_features['days_since_last_rating'] = movie_features['days_since_last_rating'].fillna(
    movie_features['days_since_last_rating'].max()
)

# Rebuild the fast dictionary lookup with these new columns
movie_features_dict = movie_features.set_index('movieId').to_dict('index')

print("New movie features added:")
print(movie_features[['movieId', 'movie_rating_count', 'movie_rating_count_log',
                        'movie_popularity_percentile', 'days_since_last_rating']].head())

New movie features added:
   movieId  movie_rating_count  movie_rating_count_log  \
0        1               49853               10.816854   
1        2               21196                9.961615   
2        3               11514                9.351406   
3        4                2479                7.816014   
4        5               11122                9.316770   

   movie_popularity_percentile  days_since_last_rating  
0                     0.999710                       0  
1                     0.996141                       0  
2                     0.989379                       0  
3                     0.947854                       4  
4                     0.988770                       1  


In [ ]:
def build_ranking_features_fast(user_id, candidate_movie_ids):
    rows = []

    if user_id in user_features_dict:
        u = user_features_dict[user_id]
        u_count, u_avg, u_std = u['user_rating_count'], u['user_avg_rating'], u['user_rating_std']
    else:
        u_count, u_avg, u_std = 0, train['rating'].mean(), 0

    for movie_id in candidate_movie_ids:
        if movie_id in movie_features_dict:
            m = movie_features_dict[movie_id]
            m_count = m['movie_rating_count']
            m_avg = m['movie_avg_rating']
            m_std = m['movie_rating_std']
            m_count_log = m['movie_rating_count_log']
            m_percentile = m['movie_popularity_percentile']
            m_recency = m['days_since_last_rating']
            genres = m['genres']
        else:
            m_count, m_avg, m_std = 0, train['rating'].mean(), 0
            m_count_log, m_percentile, m_recency = 0, 0, 9999
            genres = ""

        als_score = get_als_score(user_id, movie_id)
        genre_match = get_genre_match_score_fast(user_id, genres)

        rows.append({
            'userId': user_id, 'movieId': movie_id, 'als_score': als_score,
            'user_rating_count': u_count, 'user_avg_rating': u_avg, 'user_rating_std': u_std,
            'movie_rating_count': m_count, 'movie_avg_rating': m_avg, 'movie_rating_std': m_std,
            'movie_rating_count_log': m_count_log, 'movie_popularity_percentile': m_percentile,
            'days_since_last_rating': m_recency,
            'genre_match': genre_match,
        })

    return pd.DataFrame(rows)


# Update our feature column list to include the new ones
feature_cols = ['als_score', 'user_rating_count', 'user_avg_rating', 'user_rating_std',
                 'movie_rating_count', 'movie_avg_rating', 'movie_rating_std',
                 'movie_rating_count_log', 'movie_popularity_percentile', 'days_since_last_rating',
                 'genre_match']

print("build_ranking_features_fast updated with recency and improved popularity features.")
print("New feature_cols:", feature_cols)

build_ranking_features_fast updated with recency and improved popularity features.
New feature_cols: ['als_score', 'user_rating_count', 'user_avg_rating', 'user_rating_std', 'movie_rating_count', 'movie_avg_rating', 'movie_rating_std', 'movie_rating_count_log', 'movie_popularity_percentile', 'days_since_last_rating', 'genre_match']


In [ ]:
from sklearn.model_selection import KFold

def build_training_data_oof(sample_users=12000, k=100, n_folds=3, seed=42):
    """
    Out-of-fold candidate generation: split users into folds,
    train a separate ALS model per fold (excluding that fold's own data influence),
    generate candidates using the held-out model.
    """
    np.random.seed(seed)

    val_users = val['userId'].unique()
    eligible_users = np.array([u for u in val_users if u in user_to_idx])
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    all_rows = []

    for fold_num, (_, fold_user_positions) in enumerate(kf.split(sampled_users)):
        fold_users = sampled_users[fold_user_positions]
        print(f"\n=== Fold {fold_num + 1}/{n_folds}: {len(fold_users)} users ===")

        # Train a fresh ALS model, EXCLUDING this fold's users from the training matrix
        # This simulates "these users are new/unseen" from this model's perspective
        fold_train_mask = ~positive_train['userId'].isin(fold_users)
        fold_positive_train = positive_train[fold_train_mask]

        fold_t_user_idx = fold_positive_train['userId'].map(user_to_idx)
        fold_t_movie_idx = fold_positive_train['movieId'].map(movie_to_idx)

        fold_matrix = sparse.csr_matrix(
            (fold_positive_train['confidence'], (fold_t_user_idx, fold_t_movie_idx)),
            shape=(len(user_ids), len(movie_ids))
        )

        fold_model = AlternatingLeastSquares(factors=50, regularization=0.01, iterations=15)
        fold_model.fit(fold_matrix)

        fold_item_vectors = fold_model.item_factors.astype('float32')
        fold_index = faiss.IndexFlatIP(fold_item_vectors.shape[1])
        fold_index.add(fold_item_vectors)

        # Generate candidates for THIS fold's users using the model that never trained on them directly
        for i, user_id in enumerate(fold_users):
            if i % 500 == 0:
                print(f"  Processing user {i}/{len(fold_users)}...")

            actual_liked = set(val[(val['userId'] == user_id) & (val['rating'] >= 3.5)]['movieId'].tolist())

            if user_id not in user_to_idx:
                candidates = top_popular_movies[:k]
            else:
                u_idx = user_to_idx[user_id]
                u_vec = fold_model.user_factors[u_idx].astype('float32').reshape(1, -1)
                _, indices = fold_index.search(u_vec, k)
                candidates = [int(idx_to_movie[idx]) for idx in indices[0]]

            feature_df = build_ranking_features_fast(user_id, candidates)
            feature_df['label'] = feature_df['movieId'].apply(lambda m: 1 if m in actual_liked else 0)
            all_rows.append(feature_df)

    return pd.concat(all_rows, ignore_index=True)


training_data_oof = build_training_data_oof(sample_users=12000, k=100, n_folds=3)
print("\nOOF training data shape:", training_data_oof.shape)
print("Positive labels:", training_data_oof['label'].sum())
print("Negative labels:", (training_data_oof['label'] == 0).sum())


=== Fold 1/3: 1765 users ===


  0%|          | 0/15 [00:00<?, ?it/s]

  Processing user 0/1765...
  Processing user 500/1765...
  Processing user 1000/1765...
  Processing user 1500/1765...

=== Fold 2/3: 1765 users ===


  0%|          | 0/15 [00:00<?, ?it/s]

  Processing user 0/1765...
  Processing user 500/1765...
  Processing user 1000/1765...
  Processing user 1500/1765...

=== Fold 3/3: 1765 users ===


  0%|          | 0/15 [00:00<?, ?it/s]

  Processing user 0/1765...
  Processing user 500/1765...
  Processing user 1000/1765...
  Processing user 1500/1765...

OOF training data shape: (529500, 14)
Positive labels: 2620
Negative labels: 526880


In [ ]:
training_data_oof = training_data_oof.sort_values('userId').reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data_oof, groups=training_data_oof['userId']))

lgb_train_df = training_data_oof.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data_oof.iloc[val_idx].sort_values('userId').reset_index(drop=True)

lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

train_dataset = lgb.Dataset(lgb_train_df[feature_cols], label=lgb_train_df['label'], group=lgb_train_groups)
val_dataset = lgb.Dataset(lgb_val_df[feature_cols], label=lgb_val_df['label'], group=lgb_val_groups, reference=train_dataset)

params_oof = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_data_in_leaf': 20,
    'lambdarank_truncation_level': 10,
    'verbose': -1
}

ranker = lgb.train(
    params_oof,
    train_dataset,
    num_boost_round=400,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=20)]
)

print("LightGBM retrained on OOF data with new features.")

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': ranker.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
print(importance)

# The real test
final_precision_oof, final_ndcg_oof = evaluate_ranker_on_test(sample_users=2000)
print(f"\nFinal Precision@10 (test set, OOF + new features): {final_precision_oof:.4f}")
print(f"Final NDCG@10 (test set, OOF + new features): {final_ndcg_oof:.4f}")

print(f"\n--- Full Comparison ---")
print(f"Baseline:                Precision@10: {baseline_precision:.4f} | NDCG@10: {baseline_ndcg:.4f}")
print(f"Our system (v4, neg-sampled): Precision@10: {final_precision_v4:.4f} | NDCG@10: {final_ndcg_v4:.4f}")
print(f"Our system (v5, OOF+features): Precision@10: {final_precision_oof:.4f} | NDCG@10: {final_ndcg_oof:.4f}")

Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@10: 0.820644
Early stopping, best iteration is:
[5]	valid_0's ndcg@10: 0.824562
LightGBM retrained on OOF data with new features.
                        feature   importance
4            movie_rating_count  9268.372489
0                     als_score  5801.093521
1             user_rating_count  1526.669537
5              movie_avg_rating   903.844563
3               user_rating_std   585.798232
2               user_avg_rating   583.714848
10                  genre_match   549.137085
6              movie_rating_std   260.975012
9        days_since_last_rating    33.181691
8   movie_popularity_percentile     0.000000
7        movie_rating_count_log     0.000000
Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

Final Precision@10 (test set, OOF + new features): 0.0294
Final NDCG@10 (test set, OOF + new features): 0.0904

--- Full Comparison ---


In [ ]:
training_data_final = build_training_data_negsample(sample_users=12000, k=100, neg_ratio=5, seed=42)
print("Final training data shape:", training_data_final.shape)

Processing user 0/5295...
Processing user 500/5295...
Processing user 1000/5295...
Processing user 1500/5295...
Processing user 2000/5295...
Processing user 2500/5295...
Processing user 3000/5295...
Processing user 3500/5295...
Processing user 4000/5295...
Processing user 4500/5295...
Processing user 5000/5295...
Final training data shape: (122478, 14)


In [ ]:
training_data_final = training_data_final.sort_values('userId').reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data_final, groups=training_data_final['userId']))

lgb_train_df = training_data_final.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data_final.iloc[val_idx].sort_values('userId').reset_index(drop=True)

lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

train_dataset = lgb.Dataset(lgb_train_df[feature_cols], label=lgb_train_df['label'], group=lgb_train_groups)
val_dataset = lgb.Dataset(lgb_val_df[feature_cols], label=lgb_val_df['label'], group=lgb_val_groups, reference=train_dataset)

params_final = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_data_in_leaf': 20,
    'lambdarank_truncation_level': 10,
    'verbose': -1
}

ranker = lgb.train(
    params_final,
    train_dataset,
    num_boost_round=400,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=20)]
)

print("LightGBM reverted to negative-sampled data (with new features).")

final_precision_locked, final_ndcg_locked = evaluate_ranker_on_test(sample_users=2000)
print(f"\nLOCKED FINAL Precision@10: {final_precision_locked:.4f}")
print(f"LOCKED FINAL NDCG@10: {final_ndcg_locked:.4f}")
print(f"Baseline for comparison: Precision@10: {baseline_precision:.4f} | NDCG@10: {baseline_ndcg:.4f}")

Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@10: 0.402893
[40]	valid_0's ndcg@10: 0.404767
[60]	valid_0's ndcg@10: 0.404681
[80]	valid_0's ndcg@10: 0.406038
Early stopping, best iteration is:
[57]	valid_0's ndcg@10: 0.408526
LightGBM reverted to negative-sampled data (with new features).
Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

LOCKED FINAL Precision@10: 0.0353
LOCKED FINAL NDCG@10: 0.1136
Baseline for comparison: Precision@10: 0.2521 | NDCG@10: 0.4777


In [ ]:
def build_training_data_graded(sample_users=12000, k=100, neg_ratio=5, seed=42):
    """
    Same as negative-sampled version, but positives get GRADED relevance
    based on their actual rating (3.5=grade 1, 4.0=grade 2, 4.5=grade 3, 5.0=grade 4)
    instead of a flat label=1. Negatives stay at grade 0.
    """
    np.random.seed(seed)

    val_users = val['userId'].unique()
    eligible_users = [u for u in val_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    all_rows = []

    for i, user_id in enumerate(sampled_users):
        if i % 500 == 0:
            print(f"Processing user {i}/{len(sampled_users)}...")

        # Build a rating lookup for this user's positive val ratings
        user_val_ratings = val[(val['userId'] == user_id) & (val['rating'] >= 3.5)].set_index('movieId')['rating'].to_dict()

        candidates = get_candidates_pure_als(user_id, k=k)
        feature_df = build_ranking_features_fast(user_id, candidates)

        # Graded relevance: map rating 3.5-5.0 to grades 1-4, else 0
        def to_grade(movie_id):
            if movie_id in user_val_ratings:
                rating = user_val_ratings[movie_id]
                return int(round((rating - 3.0) * 2))  # 3.5->1, 4.0->2, 4.5->3, 5.0->4
            return 0

        feature_df['label'] = feature_df['movieId'].apply(to_grade)

        positives = feature_df[feature_df['label'] > 0]
        negatives = feature_df[feature_df['label'] == 0]

        n_pos = len(positives)
        if n_pos == 0:
            continue

        n_neg_to_keep = min(len(negatives), max(n_pos * neg_ratio, 10))
        sampled_negatives = negatives.sample(n=n_neg_to_keep, random_state=seed)

        user_final = pd.concat([positives, sampled_negatives]).sort_values('userId')
        all_rows.append(user_final)

    return pd.concat(all_rows, ignore_index=True)


training_data_graded = build_training_data_graded(sample_users=12000, k=100, neg_ratio=5)
print("\nGraded training data shape:", training_data_graded.shape)
print("Label distribution:")
print(training_data_graded['label'].value_counts().sort_index())

Processing user 0/5295...
Processing user 500/5295...
Processing user 1000/5295...
Processing user 1500/5295...
Processing user 2000/5295...
Processing user 2500/5295...
Processing user 3000/5295...
Processing user 3500/5295...
Processing user 4000/5295...
Processing user 4500/5295...
Processing user 5000/5295...

Graded training data shape: (122478, 14)
Label distribution:
label
0    100265
1      3999
2      7953
3      5403
4      4858
Name: count, dtype: int64


In [ ]:
training_data_graded = training_data_graded.sort_values('userId').reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data_graded, groups=training_data_graded['userId']))

lgb_train_df = training_data_graded.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data_graded.iloc[val_idx].sort_values('userId').reset_index(drop=True)

lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

train_dataset = lgb.Dataset(lgb_train_df[feature_cols], label=lgb_train_df['label'], group=lgb_train_groups)
val_dataset = lgb.Dataset(lgb_val_df[feature_cols], label=lgb_val_df['label'], group=lgb_val_groups, reference=train_dataset)

params_graded = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_data_in_leaf': 20,
    'lambdarank_truncation_level': 10,
    'label_gain': [0, 1, 3, 7, 15],  # exponential gain for grades 0-4, standard NDCG convention
    'verbose': -1
}

ranker = lgb.train(
    params_graded,
    train_dataset,
    num_boost_round=400,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=20)]
)

print("LightGBM retrained with graded relevance labels.")

final_precision_graded, final_ndcg_graded = evaluate_ranker_on_test(sample_users=2000)
print(f"\nFinal Precision@10 (graded relevance): {final_precision_graded:.4f}")
print(f"Final NDCG@10 (graded relevance): {final_ndcg_graded:.4f}")

print(f"\n--- Full Comparison ---")
print(f"Baseline:                    Precision@10: {baseline_precision:.4f} | NDCG@10: {baseline_ndcg:.4f}")
print(f"Our system (binary, locked): Precision@10: {final_precision_locked:.4f} | NDCG@10: {final_ndcg_locked:.4f}")
print(f"Our system (graded):         Precision@10: {final_precision_graded:.4f} | NDCG@10: {final_ndcg_graded:.4f}")

Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@10: 0.37085
[40]	valid_0's ndcg@10: 0.378338
[60]	valid_0's ndcg@10: 0.37842
[80]	valid_0's ndcg@10: 0.378606
Early stopping, best iteration is:
[61]	valid_0's ndcg@10: 0.378769
LightGBM retrained with graded relevance labels.
Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

Final Precision@10 (graded relevance): 0.0350
Final NDCG@10 (graded relevance): 0.1136

--- Full Comparison ---
Baseline:                    Precision@10: 0.2521 | NDCG@10: 0.4777
Our system (binary, locked): Precision@10: 0.0353 | NDCG@10: 0.1136
Our system (graded):         Precision@10: 0.0350 | NDCG@10: 0.1136


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build a TF-IDF representation of every movie's genres
# (treating genres like "words" in a mini "document" per movie)
movies_clean = movies.copy()
movies_clean['genres'] = movies_clean['genres'].fillna('').str.replace('|', ' ')

tfidf = TfidfVectorizer()
genre_vectors = tfidf.fit_transform(movies_clean['genres'])

# Build a lookup: movieId -> row index in this genre_vectors matrix
content_movie_to_idx = {m: i for i, m in enumerate(movies_clean['movieId'])}
content_idx_to_movie = {i: m for m, i in content_movie_to_idx.items()}

print("Genre vector matrix shape:", genre_vectors.shape)
print("Vocabulary (unique genres):", tfidf.get_feature_names_out())


def get_content_based_candidates(movie_id, k=20):
    """
    Given a movie with NO ALS history (cold-start item),
    find the k most similar movies based purely on genre overlap.
    """
    if movie_id not in content_movie_to_idx:
        return []

    idx = content_movie_to_idx[movie_id]
    sims = cosine_similarity(genre_vectors[idx], genre_vectors).flatten()

    similar_indices = sims.argsort()[::-1][1:k+1]  # skip index 0 (itself)
    return [content_idx_to_movie[i] for i in similar_indices]


# Quick test: find movies similar to a specific cold-start movie
test_coldstart_movie = list(never_in_training)[0] if len(never_in_training) > 0 else movies['movieId'].iloc[0]
similar = get_content_based_candidates(test_coldstart_movie, k=5)
print(f"\nMovie {test_coldstart_movie} genres: {movies[movies['movieId']==test_coldstart_movie]['genres'].values}")
print(f"Content-based similar movies: {similar}")
print(movies[movies['movieId'].isin(similar)][['movieId', 'title', 'genres']])

Genre vector matrix shape: (62423, 24)
Vocabulary (unique genres): ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'genres' 'horror' 'imax'
 'listed' 'musical' 'mystery' 'no' 'noir' 'romance' 'sci' 'thriller' 'war'
 'western']

Movie 196609 genres: ['Comedy|Horror']
Content-based similar movies: [12, 26684, 178045, 31909, 201318]
       movieId                                              title  \
11          12                 Dracula: Dead and Loving It (1995)   
8931     26684                               Frankenhooker (1990)   
9634     31909                                 Dr. Giggles (1992)   
49095   178045  Another Yeti a Love Story: Life on the Streets...   
59649   201318                                  Deer Woman (2005)   

              genres  
11     Comedy|Horror  
8931   Comedy|Horror  
9634   Comedy|Horror  
49095  Comedy|Horror  
59649  Comedy|Horror  


In [ ]:
def get_candidates_with_content_fallback(user_id, k=100, popular_slots=20):
    """
    Hybrid retrieval: ALS+FAISS for known items, PLUS content-based
    candidates for cold-start items the user might be interested in,
    based on genres of movies they've already liked.
    """
    if user_id not in user_to_idx:
        return top_popular_movies[:k]

    # Get normal ALS+FAISS + popularity hybrid candidates (our existing function)
    base_candidates = get_candidates(user_id, k=k - 10)  # reserve 10 slots for content-based

    # Find genres this user tends to like, based on their training-period positive ratings
    user_liked_movies = positive_train[positive_train['userId'] == user_id]['movieId'].tolist()

    if len(user_liked_movies) == 0:
        return base_candidates + top_popular_movies[:10]

    # Get content-based similar movies to what they've liked, prioritizing movies NOT in training
    # (since those are exactly the ones ALS could never find)
    content_candidates = []
    for liked_movie in user_liked_movies[:5]:  # just check their 5 most recent likes for speed
        similar = get_content_based_candidates(liked_movie, k=10)
        new_items = [m for m in similar if m not in all_training_movies]  # prioritize truly cold-start items
        content_candidates.extend(new_items)

    content_candidates = list(dict.fromkeys(content_candidates))[:10]  # dedupe, take top 10

    # Fill remaining slots with popularity if content-based didn't find enough
    if len(content_candidates) < 10:
        content_candidates += [m for m in top_popular_movies if m not in content_candidates][:10 - len(content_candidates)]

    return base_candidates + content_candidates


# Test: check recall specifically on previously-unreachable movies
def check_coldstart_recall_improvement(sample_users=1000, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    hits_before = 0
    hits_after = 0
    total_coldstart_likes = 0

    for user_id in sampled_users:
        actual_liked_raw = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        coldstart_likes = actual_liked_raw - all_training_movies  # only the previously-unreachable ones

        if len(coldstart_likes) == 0:
            continue

        total_coldstart_likes += len(coldstart_likes)

        old_candidates = set(get_candidates(user_id, k=100))
        new_candidates = set(get_candidates_with_content_fallback(user_id, k=100))

        hits_before += len(coldstart_likes & old_candidates)
        hits_after += len(coldstart_likes & new_candidates)

    print(f"Cold-start items actually liked (in sample): {total_coldstart_likes}")
    print(f"Recall on cold-start items BEFORE content fallback: {hits_before/total_coldstart_likes:.4f}")
    print(f"Recall on cold-start items AFTER content fallback: {hits_after/total_coldstart_likes:.4f}")


check_coldstart_recall_improvement(sample_users=1000)

Cold-start items actually liked (in sample): 14272
Recall on cold-start items BEFORE content fallback: 0.0000
Recall on cold-start items AFTER content fallback: 0.0008


In [ ]:
# Fix 1: properly clean genre strings (keep Sci-Fi as one token, remove the "no genres listed" placeholder)
movies_clean = movies.copy()
movies_clean['genres'] = movies_clean['genres'].replace('(no genres listed)', '')
movies_clean['genres'] = movies_clean['genres'].str.replace('Sci-Fi', 'scifi', regex=False)
movies_clean['genres'] = movies_clean['genres'].str.replace('Film-Noir', 'filmnoir', regex=False)
movies_clean['genres'] = movies_clean['genres'].fillna('').str.replace('|', ' ')

tfidf = TfidfVectorizer()
genre_vectors = tfidf.fit_transform(movies_clean['genres'])
content_movie_to_idx = {m: i for i, m in enumerate(movies_clean['movieId'])}
content_idx_to_movie = {i: m for m, i in content_movie_to_idx.items()}

print("Cleaned vocabulary:", tfidf.get_feature_names_out())

# Fix 2: use ALL of a user's liked movies (not just first 5), and check against ALL their content candidates
def get_candidates_with_content_fallback_v2(user_id, k=100):
    if user_id not in user_to_idx:
        return top_popular_movies[:k]

    base_candidates = get_candidates(user_id, k=k - 10)
    user_liked_movies = positive_train[positive_train['userId'] == user_id]['movieId'].tolist()

    if len(user_liked_movies) == 0:
        return base_candidates + top_popular_movies[:10]

    content_candidates = []
    for liked_movie in user_liked_movies:  # check ALL liked movies, not just 5
        similar = get_content_based_candidates(liked_movie, k=15)
        new_items = [m for m in similar if m not in all_training_movies]
        content_candidates.extend(new_items)

    content_candidates = list(dict.fromkeys(content_candidates))[:10]

    if len(content_candidates) < 10:
        content_candidates += [m for m in top_popular_movies if m not in content_candidates][:10 - len(content_candidates)]

    return base_candidates + content_candidates


def check_coldstart_recall_improvement_v2(sample_users=1000, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    hits_before = 0
    hits_after = 0
    total_coldstart_likes = 0

    for user_id in sampled_users:
        actual_liked_raw = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        coldstart_likes = actual_liked_raw - all_training_movies
        if len(coldstart_likes) == 0:
            continue
        total_coldstart_likes += len(coldstart_likes)

        old_candidates = set(get_candidates(user_id, k=100))
        new_candidates = set(get_candidates_with_content_fallback_v2(user_id, k=100))

        hits_before += len(coldstart_likes & old_candidates)
        hits_after += len(coldstart_likes & new_candidates)

    print(f"Cold-start items actually liked (in sample): {total_coldstart_likes}")
    print(f"Recall on cold-start items BEFORE: {hits_before/total_coldstart_likes:.4f}")
    print(f"Recall on cold-start items AFTER: {hits_after/total_coldstart_likes:.4f}")


check_coldstart_recall_improvement_v2(sample_users=1000)

Cleaned vocabulary: ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'filmnoir' 'horror' 'imax' 'musical'
 'mystery' 'romance' 'scifi' 'thriller' 'war' 'western']


KeyboardInterrupt: 

In [ ]:
# Precompute content-based candidates ONCE per unique genre string, not per movie
unique_genre_strings = movies_clean['genres'].unique()
print(f"Unique genre combinations: {len(unique_genre_strings)} (vs {len(movies_clean)} total movies)")

genre_string_to_similar_coldstart = {}

for genre_str in unique_genre_strings:
    if genre_str == '':
        continue
    idx = movies_clean[movies_clean['genres'] == genre_str].index[0]
    sims = cosine_similarity(genre_vectors[idx], genre_vectors).flatten()
    similar_indices = sims.argsort()[::-1][:30]  # top 30, we'll filter afterward
    similar_movie_ids = [content_idx_to_movie[i] for i in similar_indices]
    coldstart_only = [m for m in similar_movie_ids if m not in all_training_movies]
    genre_string_to_similar_coldstart[genre_str] = coldstart_only[:10]

# Map each movie to its genre string for fast lookup
movie_to_genre_string = movies_clean.set_index('movieId')['genres'].to_dict()

print("Precomputation done.")


def get_candidates_with_content_fallback_v3(user_id, k=100):
    if user_id not in user_to_idx:
        return top_popular_movies[:k]

    base_candidates = get_candidates(user_id, k=k - 10)
    user_liked_movies = positive_train[positive_train['userId'] == user_id]['movieId'].tolist()

    if len(user_liked_movies) == 0:
        return base_candidates + top_popular_movies[:10]

    content_candidates = []
    for liked_movie in user_liked_movies:
        genre_str = movie_to_genre_string.get(liked_movie, '')
        if genre_str in genre_string_to_similar_coldstart:
            content_candidates.extend(genre_string_to_similar_coldstart[genre_str])

    content_candidates = list(dict.fromkeys(content_candidates))[:10]

    if len(content_candidates) < 10:
        content_candidates += [m for m in top_popular_movies if m not in content_candidates][:10 - len(content_candidates)]

    return base_candidates + content_candidates

Unique genre combinations: 1639 (vs 62423 total movies)
Precomputation done.


In [ ]:
def check_coldstart_recall_improvement_v3(sample_users=1000, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    hits_before = 0
    hits_after = 0
    total_coldstart_likes = 0

    for user_id in sampled_users:
        actual_liked_raw = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        coldstart_likes = actual_liked_raw - all_training_movies
        if len(coldstart_likes) == 0:
            continue
        total_coldstart_likes += len(coldstart_likes)

        old_candidates = set(get_candidates(user_id, k=100))
        new_candidates = set(get_candidates_with_content_fallback_v3(user_id, k=100))

        hits_before += len(coldstart_likes & old_candidates)
        hits_after += len(coldstart_likes & new_candidates)

    print(f"Cold-start items actually liked (in sample): {total_coldstart_likes}")
    print(f"Recall on cold-start items BEFORE: {hits_before/total_coldstart_likes:.4f}")
    print(f"Recall on cold-start items AFTER: {hits_after/total_coldstart_likes:.4f}")


check_coldstart_recall_improvement_v3(sample_users=1000)

Cold-start items actually liked (in sample): 14272
Recall on cold-start items BEFORE: 0.0000
Recall on cold-start items AFTER: 0.0011


In [ ]:
# How many cold-start movies exist per popular genre string?
for genre_str in ['Comedy', 'Drama', 'Comedy Drama', 'Action Thriller']:
    matching = movies_clean[movies_clean['genres'] == genre_str]
    coldstart_matching = matching[~matching['movieId'].isin(all_training_movies)]
    print(f"'{genre_str}': {len(matching)} total movies, {len(coldstart_matching)} are cold-start")

'Comedy': 5674 total movies, 2699 are cold-start
'Drama': 9056 total movies, 3604 are cold-start
'Comedy Drama': 2386 total movies, 926 are cold-start
'Action Thriller': 445 total movies, 230 are cold-start


In [ ]:
# Check: are already-rated (training-period) movies showing up in our final top-10?
diag_user = [u for u in test['userId'].unique() if u in user_to_idx][0]
already_rated = set(train[train['userId'] == diag_user]['movieId'].tolist())

candidates = get_candidates(diag_user, k=100)
feature_df = build_ranking_features_fast(diag_user, candidates)
feature_df['score'] = ranker.predict(feature_df[feature_cols])
top10 = feature_df.sort_values('score', ascending=False).head(10)

already_rated_in_top10 = set(top10['movieId']) & already_rated
print(f"User {diag_user} has {len(already_rated)} movies already rated in training")
print(f"Of our final top 10, how many were ALREADY rated by this user in training? {len(already_rated_in_top10)}")
print(f"Already-rated movies in our top 10: {already_rated_in_top10}")

# Check across many users, systematically
def check_already_seen_contamination(sample_users=500, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    total_already_seen_in_top10 = 0
    total_slots = 0

    for user_id in sampled_users:
        already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())
        candidates = get_candidates(user_id, k=100)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['score'] = ranker.predict(feature_df[feature_cols])
        top10 = feature_df.sort_values('score', ascending=False).head(10)

        total_already_seen_in_top10 += len(set(top10['movieId']) & already_rated)
        total_slots += 10

    print(f"\nAcross {sample_users} users: {total_already_seen_in_top10}/{total_slots} top-10 slots ({total_already_seen_in_top10/total_slots*100:.1f}%) are movies the user ALREADY rated in training")

check_already_seen_contamination(sample_users=500)

User 48988 has 674 movies already rated in training
Of our final top 10, how many were ALREADY rated by this user in training? 5
Already-rated movies in our top 10: {7265, 1732, 114662, 102407, 128360}

Across 500 users: 2843/5000 top-10 slots (56.9%) are movies the user ALREADY rated in training


In [ ]:
def get_candidates_v_final(user_id, k=100, popular_slots=20):
    """
    Same hybrid retrieval as before, but now EXCLUDES movies the user
    already rated during training — those can never be valid future recommendations.
    """
    if user_id not in user_to_idx:
        return top_popular_movies[:k]

    already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())

    actual_popular_slots = min(popular_slots, max(k - 1, 0))
    als_k = max(k - actual_popular_slots, 1)

    user_idx = user_to_idx[user_id]
    user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)

    # Search for more than we need, since we'll filter some out
    scores, indices = index.search(user_vector, min(als_k * 3, len(idx_to_movie)))
    als_candidates_raw = [int(idx_to_movie[i]) for i in indices[0]]
    als_candidates = [m for m in als_candidates_raw if m not in already_rated][:als_k]

    popular_fill = [m for m in top_popular_movies if m not in als_candidates and m not in already_rated][:actual_popular_slots]

    return als_candidates + popular_fill


# Quick verification: confirm already-rated movies no longer appear
def check_already_seen_contamination_v2(sample_users=500, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    total_already_seen = 0
    total_candidates = 0

    for user_id in sampled_users:
        already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())
        candidates = get_candidates_v_final(user_id, k=100)
        total_already_seen += len(set(candidates) & already_rated)
        total_candidates += len(candidates)

    print(f"Already-seen movies in candidates now: {total_already_seen}/{total_candidates} ({total_already_seen/total_candidates*100:.2f}%)")

check_already_seen_contamination_v2(sample_users=500)

Already-seen movies in candidates now: 0/44543 (0.00%)


In [ ]:
# We need to update get_candidates references throughout — easiest is to just reassign
get_candidates_old = get_candidates  # keep a reference just in case
get_candidates = get_candidates_v_final

# Re-check retrieval recall with the fix
recall_final_fix = recall_at_k_positive_only(test, k=100, sample_users=2000, use_baseline=False)
print(f"Retrieval Recall@100 (already-seen excluded): {recall_final_fix:.4f}")
print(f"Previous Retrieval Recall@100: 0.0571")

# Re-run the full ranking evaluation
final_precision_v6, final_ndcg_v6 = evaluate_ranker_on_test(sample_users=2000)
print(f"\nFinal Precision@10 (already-seen excluded): {final_precision_v6:.4f}")
print(f"Final NDCG@10 (already-seen excluded): {final_ndcg_v6:.4f}")

print(f"\n--- Full Comparison ---")
print(f"Baseline:                     Precision@10: {baseline_precision:.4f} | NDCG@10: {baseline_ndcg:.4f}")
print(f"Our system (before this fix): Precision@10: {final_precision_locked:.4f} | NDCG@10: {final_ndcg_locked:.4f}")
print(f"Our system (already-seen excluded): Precision@10: {final_precision_v6:.4f} | NDCG@10: {final_ndcg_v6:.4f}")

Retrieval Recall@100 (already-seen excluded): 0.1018
Previous Retrieval Recall@100: 0.0571
Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

Final Precision@10 (already-seen excluded): 0.0643
Final NDCG@10 (already-seen excluded): 0.1878

--- Full Comparison ---
Baseline:                     Precision@10: 0.2521 | NDCG@10: 0.4777
Our system (before this fix): Precision@10: 0.0353 | NDCG@10: 0.1136
Our system (already-seen excluded): Precision@10: 0.0643 | NDCG@10: 0.1878


In [ ]:
def evaluate_baseline_on_test_fair(sample_users=2000, k_final=10, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    sampled_users = np.random.choice(test_users, size=min(sample_users, len(test_users)), replace=False)

    precisions = []
    ndcgs = []

    for user_id in sampled_users:
        actual_liked = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        if len(actual_liked) == 0:
            continue

        already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())
        baseline_top10 = [m for m in top_popular_movies if m not in already_rated][:k_final]

        hits = len(set(baseline_top10) & actual_liked)
        precisions.append(hits / k_final)

        relevance = [1 if m in actual_liked else 0 for m in baseline_top10]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        ideal_relevance = sorted(relevance, reverse=True)
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(ideal_relevance))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(precisions), np.mean(ndcgs)


baseline_precision_fair, baseline_ndcg_fair = evaluate_baseline_on_test_fair(sample_users=2000)
print(f"Baseline Precision@10 (already-seen excluded): {baseline_precision_fair:.4f}")
print(f"Baseline NDCG@10 (already-seen excluded): {baseline_ndcg_fair:.4f}")

print(f"\n--- FULLY FAIR Final Comparison ---")
print(f"Baseline (fair):    Precision@10: {baseline_precision_fair:.4f} | NDCG@10: {baseline_ndcg_fair:.4f}")
print(f"Our system (fair):  Precision@10: {final_precision_v6:.4f} | NDCG@10: {final_ndcg_v6:.4f}")

Baseline Precision@10 (already-seen excluded): 0.2580
Baseline NDCG@10 (already-seen excluded): 0.4933

--- FULLY FAIR Final Comparison ---
Baseline (fair):    Precision@10: 0.2580 | NDCG@10: 0.4933
Our system (fair):  Precision@10: 0.0643 | NDCG@10: 0.1878


In [ ]:
check_precision_ceiling(sample_users=1000)

Average relevant items available in each user's 100-candidate pool: 5.048
Users with ZERO relevant items in their pool: 23.0%
Theoretical max Precision@10 (if ranking were PERFECT): 0.3511


In [ ]:
def get_candidates_pure_als_v_final(user_id, k=100):
    """
    Pure ALS candidates for training data, now also excluding already-rated movies.
    """
    if user_id not in user_to_idx:
        return top_popular_movies[:k]

    already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())

    user_idx = user_to_idx[user_id]
    user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)
    scores, indices = index.search(user_vector, min(k * 3, len(idx_to_movie)))
    candidates_raw = [int(idx_to_movie[i]) for i in indices[0]]
    candidates = [m for m in candidates_raw if m not in already_rated][:k]

    return candidates


def build_training_data_negsample_v_final(sample_users=12000, k=100, neg_ratio=5, seed=42):
    np.random.seed(seed)
    val_users = val['userId'].unique()
    eligible_users = [u for u in val_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    all_rows = []
    for i, user_id in enumerate(sampled_users):
        if i % 1000 == 0:
            print(f"Processing user {i}/{len(sampled_users)}...")

        actual_liked = set(val[(val['userId'] == user_id) & (val['rating'] >= 3.5)]['movieId'].tolist())
        candidates = get_candidates_pure_als_v_final(user_id, k=k)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['label'] = feature_df['movieId'].apply(lambda m: 1 if m in actual_liked else 0)

        positives = feature_df[feature_df['label'] == 1]
        negatives = feature_df[feature_df['label'] == 0]
        n_pos = len(positives)
        if n_pos == 0:
            continue
        n_neg_to_keep = min(len(negatives), max(n_pos * neg_ratio, 10))
        sampled_negatives = negatives.sample(n=n_neg_to_keep, random_state=seed)
        all_rows.append(pd.concat([positives, sampled_negatives]).sort_values('userId'))

    return pd.concat(all_rows, ignore_index=True)


training_data_v_final = build_training_data_negsample_v_final(sample_users=12000, k=100, neg_ratio=5)
print("Shape:", training_data_v_final.shape)
print("Positive labels:", training_data_v_final['label'].sum())

Processing user 0/5295...
Processing user 1000/5295...
Processing user 2000/5295...
Processing user 3000/5295...
Processing user 4000/5295...
Processing user 5000/5295...
Shape: (192098, 14)
Positive labels: 41234


In [ ]:
training_data_v_final = training_data_v_final.sort_values('userId').reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data_v_final, groups=training_data_v_final['userId']))

lgb_train_df = training_data_v_final.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data_v_final.iloc[val_idx].sort_values('userId').reset_index(drop=True)

lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

train_dataset = lgb.Dataset(lgb_train_df[feature_cols], label=lgb_train_df['label'], group=lgb_train_groups)
val_dataset = lgb.Dataset(lgb_val_df[feature_cols], label=lgb_val_df['label'], group=lgb_val_groups, reference=train_dataset)

params_v_final = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_data_in_leaf': 20,
    'lambdarank_truncation_level': 10,
    'verbose': -1
}

ranker = lgb.train(
    params_v_final,
    train_dataset,
    num_boost_round=400,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=20)]
)

print("LightGBM retrained on already-seen-excluded data.")

final_precision_v7, final_ndcg_v7 = evaluate_ranker_on_test(sample_users=2000)
print(f"\nFinal Precision@10 (fully fixed): {final_precision_v7:.4f}")
print(f"Final NDCG@10 (fully fixed): {final_ndcg_v7:.4f}")

print(f"\n--- Full Comparison ---")
print(f"Baseline (fair):                Precision@10: {baseline_precision_fair:.4f} | NDCG@10: {baseline_ndcg_fair:.4f}")
print(f"Our system (already-seen fix only): Precision@10: {final_precision_v6:.4f} | NDCG@10: {final_ndcg_v6:.4f}")
print(f"Our system (fully fixed):       Precision@10: {final_precision_v7:.4f} | NDCG@10: {final_ndcg_v7:.4f}")

Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@10: 0.444003
[40]	valid_0's ndcg@10: 0.44977
[60]	valid_0's ndcg@10: 0.452455
[80]	valid_0's ndcg@10: 0.453563
[100]	valid_0's ndcg@10: 0.452423
[120]	valid_0's ndcg@10: 0.452323
Early stopping, best iteration is:
[90]	valid_0's ndcg@10: 0.455614
LightGBM retrained on already-seen-excluded data.
Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluating user 1500/2000...

Final Precision@10 (fully fixed): 0.0928
Final NDCG@10 (fully fixed): 0.2539

--- Full Comparison ---
Baseline (fair):                Precision@10: 0.2580 | NDCG@10: 0.4933
Our system (already-seen fix only): Precision@10: 0.0643 | NDCG@10: 0.1878
Our system (fully fixed):       Precision@10: 0.0928 | NDCG@10: 0.2539


In [ ]:
# Recent popularity: rating count in just the LAST 90 days of the training window
recent_cutoff = train_max_date - pd.Timedelta(days=90)
recent_ratings = train[train['datetime'] >= recent_cutoff]

recent_popularity = recent_ratings.groupby('movieId').size().reset_index()
recent_popularity.columns = ['movieId', 'recent_rating_count_90d']

movie_features = movie_features.merge(recent_popularity, on='movieId', how='left')
movie_features['recent_rating_count_90d'] = movie_features['recent_rating_count_90d'].fillna(0)
movie_features['recent_rating_count_90d_log'] = np.log1p(movie_features['recent_rating_count_90d'])

movie_features_dict = movie_features.set_index('movieId').to_dict('index')

print("Recent popularity feature added.")
print(movie_features[['movieId', 'movie_rating_count', 'recent_rating_count_90d', 'recent_rating_count_90d_log']].head())

Recent popularity feature added.
   movieId  movie_rating_count  recent_rating_count_90d  \
0        1               49853                   1049.0   
1        2               21196                    366.0   
2        3               11514                     53.0   
3        4                2479                     11.0   
4        5               11122                     86.0   

   recent_rating_count_90d_log  
0                     6.956545  
1                     5.905362  
2                     3.988984  
3                     2.484907  
4                     4.465908  


In [ ]:
def build_ranking_features_fast(user_id, candidate_movie_ids):
    rows = []

    if user_id in user_features_dict:
        u = user_features_dict[user_id]
        u_count, u_avg, u_std = u['user_rating_count'], u['user_avg_rating'], u['user_rating_std']
    else:
        u_count, u_avg, u_std = 0, train['rating'].mean(), 0

    for movie_id in candidate_movie_ids:
        if movie_id in movie_features_dict:
            m = movie_features_dict[movie_id]
            m_count = m['movie_rating_count']
            m_avg = m['movie_avg_rating']
            m_std = m['movie_rating_std']
            m_count_log = m['movie_rating_count_log']
            m_percentile = m['movie_popularity_percentile']
            m_recency = m['days_since_last_rating']
            m_recent_pop = m['recent_rating_count_90d_log']
            genres = m['genres']
        else:
            m_count, m_avg, m_std = 0, train['rating'].mean(), 0
            m_count_log, m_percentile, m_recency, m_recent_pop = 0, 0, 9999, 0
            genres = ""

        als_score = get_als_score(user_id, movie_id)
        genre_match = get_genre_match_score_fast(user_id, genres)

        rows.append({
            'userId': user_id, 'movieId': movie_id, 'als_score': als_score,
            'user_rating_count': u_count, 'user_avg_rating': u_avg, 'user_rating_std': u_std,
            'movie_rating_count': m_count, 'movie_avg_rating': m_avg, 'movie_rating_std': m_std,
            'movie_rating_count_log': m_count_log, 'movie_popularity_percentile': m_percentile,
            'days_since_last_rating': m_recency, 'recent_rating_count_90d_log': m_recent_pop,
            'genre_match': genre_match,
        })

    return pd.DataFrame(rows)


feature_cols = ['als_score', 'user_rating_count', 'user_avg_rating', 'user_rating_std',
                 'movie_rating_count', 'movie_avg_rating', 'movie_rating_std',
                 'movie_rating_count_log', 'movie_popularity_percentile', 'days_since_last_rating',
                 'recent_rating_count_90d_log', 'genre_match']

print("Feature builder updated with recent popularity. Rebuilding training data...")

training_data_v8 = build_training_data_negsample_v_final(sample_users=12000, k=100, neg_ratio=5)
print("Shape:", training_data_v8.shape)

Feature builder updated with recent popularity. Rebuilding training data...
Processing user 0/5295...
Processing user 1000/5295...
Processing user 2000/5295...
Processing user 3000/5295...
Processing user 4000/5295...
Processing user 5000/5295...
Shape: (192098, 15)


In [ ]:
training_data_v8 = training_data_v8.sort_values('userId').reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data_v8, groups=training_data_v8['userId']))

lgb_train_df = training_data_v8.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data_v8.iloc[val_idx].sort_values('userId').reset_index(drop=True)

lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

train_dataset = lgb.Dataset(lgb_train_df[feature_cols], label=lgb_train_df['label'], group=lgb_train_groups)
val_dataset = lgb.Dataset(lgb_val_df[feature_cols], label=lgb_val_df['label'], group=lgb_val_groups, reference=train_dataset)

params_v8 = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_data_in_leaf': 20,
    'lambdarank_truncation_level': 10,
    'verbose': -1
}

ranker = lgb.train(
    params_v8,
    train_dataset,
    num_boost_round=400,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=20)]
)

print("LightGBM retrained with recent-popularity feature.")

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': ranker.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
print(importance)

final_precision_v8, final_ndcg_v8 = evaluate_ranker_on_test(sample_users=2000)
print(f"\nFinal Precision@10 (with recent popularity): {final_precision_v8:.4f}")
print(f"Final NDCG@10 (with recent popularity): {final_ndcg_v8:.4f}")

print(f"\n--- Full Comparison ---")
print(f"Baseline (fair):           Precision@10: {baseline_precision_fair:.4f} | NDCG@10: {baseline_ndcg_fair:.4f}")
print(f"Our system (v7, fully fixed): Precision@10: {final_precision_v7:.4f} | NDCG@10: {final_ndcg_v7:.4f}")
print(f"Our system (v8, +recent pop): Precision@10: {final_precision_v8:.4f} | NDCG@10: {final_ndcg_v8:.4f}")

Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@10: 0.464759
[40]	valid_0's ndcg@10: 0.467544
[60]	valid_0's ndcg@10: 0.471881
[80]	valid_0's ndcg@10: 0.471782
Early stopping, best iteration is:
[62]	valid_0's ndcg@10: 0.473075
LightGBM retrained with recent-popularity feature.
                        feature    importance
10  recent_rating_count_90d_log  29210.588024
4            movie_rating_count  13928.652308
0                     als_score  13124.462397
1             user_rating_count   4698.039022
5              movie_avg_rating   4205.113263
2               user_avg_rating   2465.829443
3               user_rating_std   2337.548453
11                  genre_match   2121.747338
6              movie_rating_std   1828.417844
9        days_since_last_rating     61.442880
7        movie_rating_count_log      0.000000
8   movie_popularity_percentile      0.000000
Evaluating user 0/2000...
Evaluating user 500/2000...
Evaluating user 1000/2000...
Evaluat

In [ ]:
def evaluate_ranker_with_popularity_candidates(sample_users=2000, k_final=10, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    sampled_users = np.random.choice(test_users, size=min(sample_users, len(test_users)), replace=False)

    precisions = []
    ndcgs = []

    for user_id in sampled_users:
        actual_liked = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        if len(actual_liked) == 0:
            continue

        already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())
        # Feed the ranker the popularity baseline's own top-100 candidates instead of ALS's
        pop_candidates = [m for m in top_popular_movies if m not in already_rated][:100]

        feature_df = build_ranking_features_fast(user_id, pop_candidates)
        feature_df['score'] = ranker.predict(feature_df[feature_cols])
        top10 = feature_df.sort_values('score', ascending=False).head(k_final)
        recommended = top10['movieId'].tolist()

        hits = len(set(recommended) & actual_liked)
        precisions.append(hits / k_final)

        relevance = [1 if m in actual_liked else 0 for m in recommended]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        ideal_relevance = sorted(relevance, reverse=True)
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(ideal_relevance))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(precisions), np.mean(ndcgs)


precision_pop_candidates, ndcg_pop_candidates = evaluate_ranker_with_popularity_candidates(sample_users=2000)
print(f"Ranker given POPULARITY's top-100 as candidates: Precision@10: {precision_pop_candidates:.4f} | NDCG@10: {ndcg_pop_candidates:.4f}")
print(f"For comparison — Ranker given ALS's top-100: Precision@10: {final_precision_v8:.4f} | NDCG@10: {final_ndcg_v8:.4f}")
print(f"Popularity baseline itself: Precision@10: {baseline_precision_fair:.4f} | NDCG@10: {baseline_ndcg_fair:.4f}")

Ranker given POPULARITY's top-100 as candidates: Precision@10: 0.3313 | NDCG@10: 0.5436
For comparison — Ranker given ALS's top-100: Precision@10: 0.1001 | NDCG@10: 0.2669
Popularity baseline itself: Precision@10: 0.2580 | NDCG@10: 0.4933


In [ ]:
def get_candidates_hybrid_final(user_id, k_als=60, k_pop=60):
    """
    Blend a generous slice of ALS candidates with a generous slice of
    popularity candidates, letting the (now-proven-good) ranker pick the best 10
    from this combined, richer pool.
    """
    if user_id not in user_to_idx:
        return top_popular_movies[:k_pop]

    already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())

    user_idx = user_to_idx[user_id]
    user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)
    scores, indices = index.search(user_vector, min(k_als * 3, len(idx_to_movie)))
    als_candidates_raw = [int(idx_to_movie[i]) for i in indices[0]]
    als_candidates = [m for m in als_candidates_raw if m not in already_rated][:k_als]

    pop_candidates = [m for m in top_popular_movies if m not in already_rated and m not in als_candidates][:k_pop]

    return list(dict.fromkeys(als_candidates + pop_candidates))


def evaluate_hybrid_candidates(sample_users=2000, k_final=10, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    precisions = []
    ndcgs = []

    for user_id in sampled_users:
        actual_liked = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        if len(actual_liked) == 0:
            continue

        candidates = get_candidates_hybrid_final(user_id, k_als=60, k_pop=60)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['score'] = ranker.predict(feature_df[feature_cols])
        top10 = feature_df.sort_values('score', ascending=False).head(k_final)
        recommended = top10['movieId'].tolist()

        hits = len(set(recommended) & actual_liked)
        precisions.append(hits / k_final)

        relevance = [1 if m in actual_liked else 0 for m in recommended]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        ideal_relevance = sorted(relevance, reverse=True)
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(ideal_relevance))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(precisions), np.mean(ndcgs)


precision_hybrid, ndcg_hybrid = evaluate_hybrid_candidates(sample_users=2000)
print(f"Hybrid candidates (60 ALS + 60 popularity): Precision@10: {precision_hybrid:.4f} | NDCG@10: {ndcg_hybrid:.4f}")
print(f"Baseline: Precision@10: {baseline_precision_fair:.4f} | NDCG@10: {baseline_ndcg_fair:.4f}")

Hybrid candidates (60 ALS + 60 popularity): Precision@10: 0.0984 | NDCG@10: 0.2636
Baseline: Precision@10: 0.2580 | NDCG@10: 0.4933


In [ ]:
def diagnose_hybrid_composition(sample_users=200, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    top10_from_als = 0
    top10_from_pop = 0

    for user_id in sampled_users:
        already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())
        user_idx = user_to_idx[user_id]
        user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)
        scores, indices = index.search(user_vector, min(180, len(idx_to_movie)))
        als_candidates_raw = [int(idx_to_movie[i]) for i in indices[0]]
        als_set = set([m for m in als_candidates_raw if m not in already_rated][:60])
        pop_set = set([m for m in top_popular_movies if m not in already_rated and m not in als_set][:60])

        candidates = list(als_set) + list(pop_set)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['score'] = ranker.predict(feature_df[feature_cols])
        top10 = feature_df.sort_values('score', ascending=False).head(10)

        for m in top10['movieId']:
            if m in als_set:
                top10_from_als += 1
            elif m in pop_set:
                top10_from_pop += 1

    print(f"Of final top-10 picks: {top10_from_als} came from ALS candidates, {top10_from_pop} came from popularity candidates")


diagnose_hybrid_composition(sample_users=200)

Of final top-10 picks: 1736 came from ALS candidates, 264 came from popularity candidates


In [ ]:
def build_training_data_hybrid(sample_users=12000, k_als=60, k_pop=60, neg_ratio=5, seed=42):
    np.random.seed(seed)
    val_users = val['userId'].unique()
    eligible_users = [u for u in val_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    all_rows = []
    for i, user_id in enumerate(sampled_users):
        if i % 1000 == 0:
            print(f"Processing user {i}/{len(sampled_users)}...")

        actual_liked = set(val[(val['userId'] == user_id) & (val['rating'] >= 3.5)]['movieId'].tolist())
        candidates = get_candidates_hybrid_final(user_id, k_als=k_als, k_pop=k_pop)

        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['label'] = feature_df['movieId'].apply(lambda m: 1 if m in actual_liked else 0)

        positives = feature_df[feature_df['label'] == 1]
        negatives = feature_df[feature_df['label'] == 0]
        n_pos = len(positives)
        if n_pos == 0:
            continue
        n_neg_to_keep = min(len(negatives), max(n_pos * neg_ratio, 10))
        sampled_negatives = negatives.sample(n=n_neg_to_keep, random_state=seed)
        all_rows.append(pd.concat([positives, sampled_negatives]).sort_values('userId'))

    return pd.concat(all_rows, ignore_index=True)


training_data_hybrid = build_training_data_hybrid(sample_users=12000, k_als=60, k_pop=60, neg_ratio=5)
print("Shape:", training_data_hybrid.shape)
print("Positive labels:", training_data_hybrid['label'].sum())

Processing user 0/5295...
Processing user 1000/5295...
Processing user 2000/5295...
Processing user 3000/5295...
Processing user 4000/5295...
Processing user 5000/5295...
Shape: (176326, 15)
Positive labels: 36052


In [ ]:
training_data_hybrid = training_data_hybrid.sort_values('userId').reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(training_data_hybrid, groups=training_data_hybrid['userId']))

lgb_train_df = training_data_hybrid.iloc[train_idx].sort_values('userId').reset_index(drop=True)
lgb_val_df = training_data_hybrid.iloc[val_idx].sort_values('userId').reset_index(drop=True)

lgb_train_groups = lgb_train_df.groupby('userId').size().values
lgb_val_groups = lgb_val_df.groupby('userId').size().values

train_dataset = lgb.Dataset(lgb_train_df[feature_cols], label=lgb_train_df['label'], group=lgb_train_groups)
val_dataset = lgb.Dataset(lgb_val_df[feature_cols], label=lgb_val_df['label'], group=lgb_val_groups, reference=train_dataset)

params_hybrid = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_data_in_leaf': 20,
    'lambdarank_truncation_level': 10,
    'verbose': -1
}

ranker = lgb.train(
    params_hybrid,
    train_dataset,
    num_boost_round=400,
    valid_sets=[val_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=20)]
)

print("LightGBM retrained on matched hybrid distribution.")

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': ranker.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
print(importance)

final_precision_hybrid_trained, final_ndcg_hybrid_trained = evaluate_hybrid_candidates(sample_users=2000)
print(f"\nFinal Precision@10 (hybrid-trained ranker): {final_precision_hybrid_trained:.4f}")
print(f"Final NDCG@10 (hybrid-trained ranker): {final_ndcg_hybrid_trained:.4f}")

print(f"\n--- Full Comparison ---")
print(f"Baseline:                      Precision@10: {baseline_precision_fair:.4f} | NDCG@10: {baseline_ndcg_fair:.4f}")
print(f"Our system (v8, ALS-only trained): Precision@10: {final_precision_v8:.4f} | NDCG@10: {final_ndcg_v8:.4f}")
print(f"Our system (hybrid-trained):   Precision@10: {final_precision_hybrid_trained:.4f} | NDCG@10: {final_ndcg_hybrid_trained:.4f}")

Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@10: 0.493519
[40]	valid_0's ndcg@10: 0.500872
[60]	valid_0's ndcg@10: 0.499835
[80]	valid_0's ndcg@10: 0.498628
Early stopping, best iteration is:
[51]	valid_0's ndcg@10: 0.501628
LightGBM retrained on matched hybrid distribution.
                        feature    importance
10  recent_rating_count_90d_log  26107.670531
0                     als_score  22733.603640
4            movie_rating_count  12791.808254
1             user_rating_count   4097.032074
5              movie_avg_rating   2849.998985
6              movie_rating_std   2187.712797
3               user_rating_std   2004.391923
2               user_avg_rating   1899.463615
11                  genre_match   1434.660766
9        days_since_last_rating    245.164958
7        movie_rating_count_log      0.000000
8   movie_popularity_percentile      0.000000

Final Precision@10 (hybrid-trained ranker): 0.0985
Final NDCG@10 (hybrid-trained ranker):

In [ ]:
def get_candidates_hybrid_tuned(user_id, k_als=30, k_pop=70):
    if user_id not in user_to_idx:
        return top_popular_movies[:k_pop]
    already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())
    user_idx = user_to_idx[user_id]
    user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)
    scores, indices = index.search(user_vector, min(k_als * 3, len(idx_to_movie)))
    als_candidates_raw = [int(idx_to_movie[i]) for i in indices[0]]
    als_candidates = [m for m in als_candidates_raw if m not in already_rated][:k_als]
    pop_candidates = [m for m in top_popular_movies if m not in already_rated and m not in als_candidates][:k_pop]
    return list(dict.fromkeys(als_candidates + pop_candidates))

def evaluate_hybrid_tuned(sample_users=2000, k_als=30, k_pop=70, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)
    precisions, ndcgs = [], []
    for user_id in sampled_users:
        actual_liked = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        if len(actual_liked) == 0: continue
        candidates = get_candidates_hybrid_tuned(user_id, k_als, k_pop)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['score'] = ranker.predict(feature_df[feature_cols])
        top10 = feature_df.sort_values('score', ascending=False).head(10)
        recommended = top10['movieId'].tolist()
        hits = len(set(recommended) & actual_liked)
        precisions.append(hits / 10)
        relevance = [1 if m in actual_liked else 0 for m in recommended]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(sorted(relevance, reverse=True)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(precisions), np.mean(ndcgs)

p70, n70 = evaluate_hybrid_tuned(sample_users=2000, k_als=30, k_pop=70)
print(f"30 ALS + 70 pop: Precision@10: {p70:.4f} | NDCG@10: {n70:.4f}")

30 ALS + 70 pop: Precision@10: 0.0967 | NDCG@10: 0.2623


In [ ]:
precision_pop_check, ndcg_pop_check = evaluate_ranker_with_popularity_candidates(sample_users=2000)
print(f"CURRENT (hybrid-trained) ranker given POPULARITY's top-100: Precision@10: {precision_pop_check:.4f} | NDCG@10: {ndcg_pop_check:.4f}")
print(f"Earlier (ALS-only-trained) ranker on same test: Precision@10: 0.3268 | NDCG@10: 0.5209")

CURRENT (hybrid-trained) ranker given POPULARITY's top-100: Precision@10: 0.3386 | NDCG@10: 0.5496
Earlier (ALS-only-trained) ranker on same test: Precision@10: 0.3268 | NDCG@10: 0.5209


In [ ]:
def diagnose_hybrid_composition_v2(sample_users=200, k_als=30, k_pop=70, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    top10_from_als = 0
    top10_from_pop = 0

    for user_id in sampled_users:
        already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())
        user_idx = user_to_idx[user_id]
        user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)
        scores, indices = index.search(user_vector, min(k_als * 3, len(idx_to_movie)))
        als_candidates_raw = [int(idx_to_movie[i]) for i in indices[0]]
        als_set = set([m for m in als_candidates_raw if m not in already_rated][:k_als])
        pop_set = set([m for m in top_popular_movies if m not in already_rated and m not in als_set][:k_pop])

        candidates = list(als_set) + list(pop_set)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['score'] = ranker.predict(feature_df[feature_cols])
        top10 = feature_df.sort_values('score', ascending=False).head(10)

        for m in top10['movieId']:
            if m in als_set:
                top10_from_als += 1
            elif m in pop_set:
                top10_from_pop += 1

    print(f"Of final top-10 picks (30 ALS/70 pop pool): {top10_from_als} came from ALS, {top10_from_pop} came from popularity")


diagnose_hybrid_composition_v2(sample_users=200, k_als=30, k_pop=70)

Of final top-10 picks (30 ALS/70 pop pool): 1500 came from ALS, 495 came from popularity


In [ ]:
feature_cols_no_als = [c for c in feature_cols if c != 'als_score']
print("Features without als_score:", feature_cols_no_als)

train_dataset_no_als = lgb.Dataset(lgb_train_df[feature_cols_no_als], label=lgb_train_df['label'], group=lgb_train_groups)
val_dataset_no_als = lgb.Dataset(lgb_val_df[feature_cols_no_als], label=lgb_val_df['label'], group=lgb_val_groups, reference=train_dataset_no_als)

ranker_no_als = lgb.train(
    params_hybrid,
    train_dataset_no_als,
    num_boost_round=400,
    valid_sets=[val_dataset_no_als],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=20)]
)

print("LightGBM retrained WITHOUT als_score.")


def evaluate_hybrid_no_als(sample_users=2000, k_als=30, k_pop=70, seed=42):
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)
    precisions, ndcgs = [], []
    for user_id in sampled_users:
        actual_liked = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        if len(actual_liked) == 0: continue
        candidates = get_candidates_hybrid_tuned(user_id, k_als, k_pop)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['score'] = ranker_no_als.predict(feature_df[feature_cols_no_als])
        top10 = feature_df.sort_values('score', ascending=False).head(10)
        recommended = top10['movieId'].tolist()
        hits = len(set(recommended) & actual_liked)
        precisions.append(hits / 10)
        relevance = [1 if m in actual_liked else 0 for m in recommended]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(sorted(relevance, reverse=True)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(precisions), np.mean(ndcgs)


p_no_als, n_no_als = evaluate_hybrid_no_als(sample_users=2000, k_als=30, k_pop=70)
print(f"\nWithout als_score (30 ALS/70 pop pool): Precision@10: {p_no_als:.4f} | NDCG@10: {n_no_als:.4f}")
print(f"With als_score (same pool): Precision@10: 0.0990 | NDCG@10: 0.2596")
print(f"Baseline: Precision@10: {baseline_precision_fair:.4f} | NDCG@10: {baseline_ndcg_fair:.4f}")

Features without als_score: ['user_rating_count', 'user_avg_rating', 'user_rating_std', 'movie_rating_count', 'movie_avg_rating', 'movie_rating_std', 'movie_rating_count_log', 'movie_popularity_percentile', 'days_since_last_rating', 'recent_rating_count_90d_log', 'genre_match']
Training until validation scores don't improve for 30 rounds
[20]	valid_0's ndcg@10: 0.476344
[40]	valid_0's ndcg@10: 0.477692
[60]	valid_0's ndcg@10: 0.480515
[80]	valid_0's ndcg@10: 0.478192
Early stopping, best iteration is:
[63]	valid_0's ndcg@10: 0.481749
LightGBM retrained WITHOUT als_score.

Without als_score (30 ALS/70 pop pool): Precision@10: 0.0873 | NDCG@10: 0.2446
With als_score (same pool): Precision@10: 0.0990 | NDCG@10: 0.2596
Baseline: Precision@10: 0.2580 | NDCG@10: 0.4933


In [ ]:
def compute_coverage_and_diversity(recommend_fn, sample_users=1000, k_final=10, seed=42, label=""):
    """
    Coverage: what fraction of the ENTIRE catalog ever gets recommended across users?
    Diversity: on average, how many distinct genres appear in a single user's top-10?
    """
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    all_recommended_movies = set()
    genre_diversity_scores = []

    for user_id in sampled_users:
        recommended = recommend_fn(user_id)
        all_recommended_movies.update(recommended)

        genres_seen = set()
        for movie_id in recommended:
            if movie_id in movie_features_dict:
                g = movie_features_dict[movie_id]['genres']
                if g:
                    genres_seen.update(g.split('|'))
        genre_diversity_scores.append(len(genres_seen))

    catalog_size = len(movies_clean)
    coverage = len(all_recommended_movies) / catalog_size
    avg_diversity = np.mean(genre_diversity_scores)

    print(f"--- {label} ---")
    print(f"Catalog coverage: {coverage:.4f} ({len(all_recommended_movies)} unique movies recommended out of {catalog_size})")
    print(f"Avg genres per user's top-10: {avg_diversity:.2f}")
    return coverage, avg_diversity


# Popularity baseline: same fixed top-10 for everyone
def popularity_recommend(user_id):
    already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())
    return [m for m in top_popular_movies if m not in already_rated][:10]

# Our system: ALS-only candidates through the ranker
def our_system_recommend(user_id):
    candidates = get_candidates_v_final(user_id, k=100)
    feature_df = build_ranking_features_fast(user_id, candidates)
    feature_df['score'] = ranker.predict(feature_df[feature_cols])
    return feature_df.sort_values('score', ascending=False).head(10)['movieId'].tolist()


pop_coverage, pop_diversity = compute_coverage_and_diversity(popularity_recommend, sample_users=1000, label="Popularity Baseline")
our_coverage, our_diversity = compute_coverage_and_diversity(our_system_recommend, sample_users=1000, label="Our System (ALS+Ranker)")

--- Popularity Baseline ---
Catalog coverage: 0.0016 (100 unique movies recommended out of 62423)
Avg genres per user's top-10: 11.75
--- Our System (ALS+Ranker) ---
Catalog coverage: 0.0119 (742 unique movies recommended out of 62423)
Avg genres per user's top-10: 11.36


In [ ]:
def evaluate_als_only(sample_users=2000, k_final=10, seed=42):
    """
    Pure ALS+FAISS, taking its top-10 directly (no ranking stage at all) —
    already-seen movies excluded, same fairness standard as everything else.
    """
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    precisions, ndcgs = [], []
    for user_id in sampled_users:
        actual_liked = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        if len(actual_liked) == 0: continue

        # ALS's own top-10, no ranker involved — just raw FAISS similarity order
        already_rated = set(train[train['userId'] == user_id]['movieId'].tolist())
        user_idx = user_to_idx[user_id]
        user_vector = model.user_factors[user_idx].astype('float32').reshape(1, -1)
        scores, indices = index.search(user_vector, min(30, len(idx_to_movie)))
        als_raw = [int(idx_to_movie[i]) for i in indices[0]]
        recommended = [m for m in als_raw if m not in already_rated][:k_final]

        hits = len(set(recommended) & actual_liked)
        precisions.append(hits / k_final)
        relevance = [1 if m in actual_liked else 0 for m in recommended]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(sorted(relevance, reverse=True)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(precisions), np.mean(ndcgs)


als_only_precision, als_only_ndcg = evaluate_als_only(sample_users=2000)

print("=== FINAL RESULTS TABLE ===\n")
results_table = pd.DataFrame({
    'Configuration': ['Popularity Baseline', 'ALS Only (no ranker)', 'ALS + Ranker', 'Popularity Candidates + Ranker'],
    'Precision@10': [baseline_precision_fair, als_only_precision, final_precision_v8, 0.3409],
    'NDCG@10': [baseline_ndcg_fair, als_only_ndcg, final_ndcg_v8, 0.5553],
})
print(results_table.to_string(index=False))

=== FINAL RESULTS TABLE ===

                 Configuration  Precision@10  NDCG@10
           Popularity Baseline      0.258011 0.493276
          ALS Only (no ranker)      0.066734 0.198252
                  ALS + Ranker      0.100101 0.266872
Popularity Candidates + Ranker      0.340900 0.555300


In [ ]:
def evaluate_warm_only(sample_users=2000, k_final=10, seed=42):
    """
    Same as our main evaluation, but ground truth is restricted to
    'warm' items only — movies that existed in training (all_training_movies).
    This isolates ALS's true performance from the unfixable cold-item gap.
    """
    np.random.seed(seed)
    test_users = test['userId'].unique()
    eligible_users = [u for u in test_users if u in user_to_idx]
    sampled_users = np.random.choice(eligible_users, size=min(sample_users, len(eligible_users)), replace=False)

    precisions, ndcgs = [], []
    for user_id in sampled_users:
        actual_liked_raw = set(test[(test['userId'] == user_id) & (test['rating'] >= 3.5)]['movieId'].tolist())
        actual_liked = actual_liked_raw & all_training_movies  # WARM items only
        if len(actual_liked) == 0: continue

        candidates = get_candidates_v_final(user_id, k=100)
        feature_df = build_ranking_features_fast(user_id, candidates)
        feature_df['score'] = ranker.predict(feature_df[feature_cols])
        top10 = feature_df.sort_values('score', ascending=False).head(k_final)
        recommended = top10['movieId'].tolist()

        hits = len(set(recommended) & actual_liked)
        precisions.append(hits / k_final)
        relevance = [1 if m in actual_liked else 0 for m in recommended]
        dcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevance))
        idcg = sum(rel / np.log2(idx + 2) for idx, rel in enumerate(sorted(relevance, reverse=True)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(precisions), np.mean(ndcgs)


warm_precision, warm_ndcg = evaluate_warm_only(sample_users=2000)
print(f"Fraction of test positives that are cold items: 0.382")
print(f"Precision@10 (WARM items only, current final model): {warm_precision:.4f}")
print(f"NDCG@10 (WARM items only, current final model): {warm_ndcg:.4f}")
print(f"\nFor comparison — Precision@10 (all items, including unreachable cold ones): {final_precision_v8:.4f}")

Fraction of test positives that are cold items: 0.382
Precision@10 (WARM items only, current final model): 0.1021
NDCG@10 (WARM items only, current final model): 0.2780

For comparison — Precision@10 (all items, including unreachable cold ones): 0.1001


In [ ]:
import os
import pickle
import json

# Create a folder for artifacts (in Colab's local storage first, we'll move to Drive after)
os.makedirs('artifacts', exist_ok=True)

# 1. ALS model factors
np.savez('artifacts/als_model.npz',
         user_factors=model.user_factors,
         item_factors=model.item_factors)

# 2. FAISS index
faiss.write_index(index, 'artifacts/faiss_index.bin')

# 3. LightGBM ranker
ranker.save_model('artifacts/lightgbm_ranker.txt')

# 4. Feature tables (as Parquet, per our original architecture plan)
user_features.to_parquet('artifacts/user_features.parquet')
movie_features.to_parquet('artifacts/movie_features.parquet')

# 5. Popularity fallback list
with open('artifacts/top_popular_movies.json', 'w') as f:
    json.dump(top_popular_movies, f)

# 6. ID mappings (needed to translate between real IDs and matrix positions)
mappings = {
    'user_to_idx': user_to_idx,
    'movie_to_idx': movie_to_idx,
    'idx_to_movie': idx_to_movie,
    'feature_cols': feature_cols,
}
with open('artifacts/mappings.pkl', 'wb') as f:
    pickle.dump(mappings, f)

# 7. Genre affinity table (for genre_match feature) and content-based similarity data
user_genre_affinity.to_parquet('artifacts/user_genre_affinity.parquet')

# 8. A small subset of movies.csv (just titles/genres, needed for display + genre matching)
movies_clean[['movieId', 'title', 'genres']].to_parquet('artifacts/movies_metadata.parquet')

# List everything we saved
print("Artifacts saved:")
for f in os.listdir('artifacts'):
    size_mb = os.path.getsize(f'artifacts/{f}') / (1024*1024)
    print(f"  {f}: {size_mb:.2f} MB")

Artifacts saved:
  user_genre_affinity.parquet: 4.92 MB
  als_model.npz: 32.87 MB
  movie_features.parquet: 1.62 MB
  mappings.pkl: 4.02 MB
  top_popular_movies.json: 0.00 MB
  faiss_index.bin: 6.57 MB
  user_features.parquet: 2.67 MB
  movies_metadata.parquet: 1.68 MB
  lightgbm_ranker.txt: 0.34 MB


In [ ]:
import shutil

shutil.make_archive('recsys_artifacts', 'zip', 'artifacts')
print("Zipped successfully.")

from google.colab import files
files.download('recsys_artifacts.zip')

Zipped successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('recsys_artifacts.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>